# 多平台电商存量经营分析 · 自动化分析流水线

**业务目标**：同时运营淘宝/京东/抖音的电商，需要①定位各平台转化流失卡点；②建立标准化用户价值分层；③识别滞销/爆品 SKU。

**六阶段流程**：数据质量校验 → SQL 多表关联与基础指标 → Cohort/RFM/帕累托/商品分层 → EDA 特征洞察 → BI 图表与看板数据 → 结论与产物清单。

**统一口径**
- 有效订单：`order_status = '已完成'`；GMV：有效订单 `total_amount` 之和；快照日 2026-12-31。
- 金额单位元，用户/订单/SKU 一律按全局 ID 去重。

**技术边界（硬性）**：不使用 PCA/K-Means，不做 A/B 实验；仅使用一次默认参数逻辑回归做下单转化的特征方向验证（训练集 AUC = 0.78，并附 5 组基线对照 0.5124~0.7766；只取系数方向与漏斗/EDA 交叉印证，不做调参、无交叉验证、不用于预测上线）。模块 24B/24C 对订单状态、促销等字段使用卡方/Spearman 观察性独立性检验，模块 28 对品牌/支付方式/配送方式/性别/停留时长做同一套卡方 + Cramer's V 筛查，均无实验分组、不做因果解读。

**输入**：`data/` 下 9 张业务 CSV；原始数据集随附的 `auth_user.csv` 含 PBKDF2 密码哈希，已按敏感信息从项目删除，**全程不读取**。
**输出**：`output/metrics/`（SQL 与统计中间结果）、`output/bi/`（PowerBI 聚合集）、`figures/`（中文图表）。

## 模块 0｜环境准备与全局配置
本格目标：导入依赖、统一路径与快照日、配置中文字体（防止图表中文乱码）。

In [1]:
import os
import re
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ---- 路径自适应：无论 cwd 是项目根目录还是 code/ 都能定位 data/ ----
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
CODE = ROOT / "code"
FIG = ROOT / "figures"
MET = ROOT / "output" / "metrics"
BIDIR = ROOT / "output" / "bi"
for _d in (FIG, MET, BIDIR):
    _d.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DATE = "2026-12-31"   # R 与 Cohort 窗口完整性判断的统计快照日
BUSINESS_FILES = ["platform", "order_status", "payment_method", "shipping_method",
                  "user", "product", "user_behavior", "order", "order_item"]
print("项目根目录：", ROOT)

# ---- 中文字体（Windows 优先微软雅黑，回退黑体），保证图表无乱码 ----
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid", {"font.sans-serif": ["Microsoft YaHei", "SimHei"]})

项目根目录： D:\ecom_analysis


## 模块 1｜读取 9 张业务 CSV（显式跳过 auth_user.csv）
本格目标：只加载业务表，输出每张表的行列规模与样例，建立数据第一印象。

In [2]:
# auth_user.csv 是后台管理员账号表，按项目要求全程不读取、不分析
raw = {n: pd.read_csv(DATA / f"{n}.csv") for n in BUSINESS_FILES}
platform, order_status, payment_method, shipping_method = (raw[n] for n in
    ["platform", "order_status", "payment_method", "shipping_method"])
user, product, user_behavior, order, order_item = (raw[n] for n in
    ["user", "product", "user_behavior", "order", "order_item"])

# 时间字段转 datetime（转换失败数也是质量检查项，后续模块验证为 0）
order["order_time"] = pd.to_datetime(order["order_time"], errors="coerce")
order["payment_time"] = pd.to_datetime(order["payment_time"], errors="coerce")
user_behavior["behavior_time"] = pd.to_datetime(user_behavior["behavior_time"], errors="coerce")
user["registration_date"] = pd.to_datetime(user["registration_date"], errors="coerce")

# 数据规模总览
pd.DataFrame({"表名": BUSINESS_FILES,
              "行数": [len(raw[n]) for n in BUSINESS_FILES],
              "列数": [raw[n].shape[1] for n in BUSINESS_FILES]})

,表名,行数,列数
0,platform,5,5
1,order_status,5,6
2,payment_method,5,5
3,shipping_method,4,5
4,user,5486,11
5,product,3298,10
6,user_behavior,21998,15
7,order,10996,15
8,order_item,27523,7


## 模块 2｜数据质量校验（一）：缺失值
本格目标：逐表逐字段统计缺失数量与占比；只呈现事实，不做删除。结果落盘 `output/metrics/dq_missing.csv`。

In [3]:
miss_rows = []
for name, df in raw.items():
    for col in df.columns:
        cnt = int(df[col].isna().sum())
        miss_rows.append({"表名": name, "字段": col, "数据类型": str(df[col].dtype),
                          "缺失数": cnt, "缺失占比_%": round(cnt / len(df) * 100, 2)})
dq_missing = pd.DataFrame(miss_rows)
dq_missing.to_csv(MET / "dq_missing.csv", index=False, encoding="utf-8-sig")

# 仅展示存在缺失的字段（其余字段缺失数全部为 0）
dq_missing[dq_missing["缺失数"] > 0]

,表名,字段,数据类型,缺失数,缺失占比_%
41,product,tags,float64,3298,100.0000
51,user_behavior,referrer,object,7158,32.5400
53,user_behavior,app_version,float64,21998,100.0000
70,order,coupon_code,object,5520,50.2000


## 模块 3｜数据质量校验（二）：主键重复与完全重复行
本格目标：验证各表主键唯一性，并检查是否存在整行重复数据。

In [4]:
PK_MAP = {"platform": "code", "order_status": "code", "payment_method": "code",
          "shipping_method": "code", "user": "global_user_id",
          "product": "global_product_id", "user_behavior": "behavior_id",
          "order": "order_id", "order_item": "order_item_id"}

dup_rows = []
for name, df in raw.items():
    dup_rows.append({"表名": name, "主键": PK_MAP[name],
                     "主键重复行数": int(df[PK_MAP[name]].duplicated().sum()),
                     "完全重复行数": int(df.duplicated().sum())})
dq_duplicate = pd.DataFrame(dup_rows)
dq_duplicate.to_csv(MET / "dq_duplicate.csv", index=False, encoding="utf-8-sig")
dq_duplicate

,表名,主键,主键重复行数,完全重复行数
0,platform,code,0,0
1,order_status,code,0,0
2,payment_method,code,0,0
3,shipping_method,code,0,0
4,user,global_user_id,0,0
5,product,global_product_id,0,0
6,user_behavior,behavior_id,0,0
7,order,order_id,0,0
8,order_item,order_item_id,0,0


## 模块 4｜数据质量校验（三）：外键完整性
本格目标：统计行为表、订单表、明细表引用的用户/商品/订单 ID 在主表中匹配不上的孤儿记录数。

In [5]:
def orphan_count(child, col, parent, pcol):
    """返回 (子表记录数, 无法匹配主表的孤儿记录数)"""
    sub = raw[child][col].dropna()
    bad = ~sub.isin(set(raw[parent][pcol].dropna()))
    return len(sub), int(bad.sum())

fk_checks = [
    ("user_behavior", "global_user_id", "user", "global_user_id", "行为→用户"),
    ("user_behavior", "global_product_id", "product", "global_product_id", "行为→商品"),
    ("order", "global_user_id", "user", "global_user_id", "订单→用户"),
    ("order_item", "order_id", "order", "order_id", "明细→订单"),
    ("order_item", "global_product_id", "product", "global_product_id", "明细→商品"),
]
dq_fk = pd.DataFrame([{"校验关系": label, "子表记录数": orphan_count(c, cc, p, pc)[0],
                       "孤儿记录数": orphan_count(c, cc, p, pc)[1]}
                      for c, cc, p, pc, label in fk_checks])
dq_fk.to_csv(MET / "dq_fk.csv", index=False, encoding="utf-8-sig")
dq_fk

,校验关系,子表记录数,孤儿记录数
0,行为→用户,21998,0
1,行为→商品,21998,0
2,订单→用户,10996,0
3,明细→订单,27523,0
4,明细→商品,27523,0


## 模块 5｜数据质量校验（四）：业务逻辑异常

本格目标：按提示词逐项检出时序与金额异常，遵循「**先统计数量/占比，再标记保留或过滤，写明理由，绝不盲目删除**」原则。异常台账落盘 `output/metrics/dq_anomaly.csv`。

In [6]:
anom_rows = []
def add_anom(kind, cnt, total, choice, reason):
    pct = round(cnt / total * 100, 2) if total else 0.0
    anom_rows.append({"异常类型": kind, "异常样本数": cnt, "基数": total,
                      "占比_%": pct, "处理选择": choice, "选择理由": reason})

# --- ① 订单时间早于注册日期（时序逻辑脏数据）---
om = order.merge(user[["global_user_id", "registration_date"]], on="global_user_id", how="left")
bad_order_reg = int((om.order_time < om.registration_date).sum())
add_anom("订单下单时间早于用户注册日期", bad_order_reg, len(order), "标记保留，不删除",
         "占比50.01%为两日期独立随机所致；删除会破坏平台/等级结构。Cohort中注册前订单offset为负不计入留存；GMV/RFM不依赖注册日期，不受影响")

# --- ② 行为时间早于注册日期 ---
bm = user_behavior.merge(user[["global_user_id", "registration_date"]], on="global_user_id", how="left")
bad_beh_reg = int((bm.behavior_time < bm.registration_date).sum())
add_anom("行为时间早于用户注册日期", bad_beh_reg, len(user_behavior), "标记保留，不删除",
         "漏斗按行为类型对用户去重，不使用注册日期，结论不受影响")

# --- ③ 用户×商品粒度：首次下单早于首次浏览 ---
br = (user_behavior[user_behavior.behavior_type == "浏览"]
      .groupby(["global_user_id", "global_product_id"]).behavior_time.min())
od = (user_behavior[user_behavior.behavior_type == "下单"]
      .groupby(["global_user_id", "global_product_id"]).behavior_time.min())
pair = pd.concat([br.rename("t_browse"), od.rename("t_order")], axis=1).dropna()
bad_pair = int((pair.t_order < pair.t_browse).sum())
add_anom("用户×商品粒度下单早于浏览", bad_pair, len(pair), "标记保留，不删除",
         f"占双行为对{round(bad_pair/len(pair)*100,2)}%，量级极低；漏斗不依赖同用户同商品时序，指标不受影响")

# --- ④ 同一会话被多个用户共用（疑似串号）---
sess_user_n = user_behavior.groupby("session_id").global_user_id.nunique()
bad_sess = int((sess_user_n > 1).sum())
add_anom("同一会话出现多个用户(疑似串号)", bad_sess, len(sess_user_n), "标记保留，不删除",
         "占比0.009%可忽略；漏斗按global_user_id去重，串号不产生重复计数")

# --- ⑤ 金额/数量类规则（本数据实际检出均为 0，仍留痕）---
add_anom("订单total_amount为负", int((order.total_amount < 0).sum()), len(order), "无需处理", "检出0条")
add_anom("订单discount_amount为负", int((order.discount_amount < 0).sum()), len(order), "无需处理", "检出0条")
add_anom("支付时间早于下单时间",
         int(((order.payment_time.notna()) & (order.order_time.notna()) &
              (order.payment_time < order.order_time)).sum()), len(order), "无需处理", "检出0条")
add_anom("明细quantity<=0", int((order_item.quantity <= 0).sum()), len(order_item), "无需处理", "检出0条（数量恒为1）")
add_anom("明细item_total为负", int((order_item.item_total < 0).sum()), len(order_item), "无需处理", "检出0条")
_mismatch = int(((order_item.quantity * order_item.unit_price - order_item.item_total).abs() > 0.02).sum())
add_anom("item_total不等于数量×单价", _mismatch, len(order_item), "无需处理", "检出0条，27523行全部精确勾稽")

dq_anomaly = pd.DataFrame(anom_rows)
dq_anomaly.to_csv(MET / "dq_anomaly.csv", index=False, encoding="utf-8-sig")
dq_anomaly

,异常类型,异常样本数,基数,占比_%,处理选择,选择理由
0,订单下单时间早于用户注册日期,5499,10996,50.0100,标记保留，不删除,占比50.01%为模拟数据两日期独立随机所致；删除会破坏平台/等级结构。Cohort中注册前...
1,行为时间早于用户注册日期,11046,21998,50.2100,标记保留，不删除,漏斗按行为类型对用户去重，不使用注册日期，结论不受影响
2,用户×商品粒度下单早于浏览,12,1087,1.1000,标记保留，不删除,占双行为对1.1%，量级极低；漏斗不依赖同用户同商品时序，指标不受影响
3,同一会话出现多个用户(疑似串号),2,21996,0.0100,标记保留，不删除,占比0.009%可忽略；漏斗按global_user_id去重，串号不产生重复计数
4,订单total_amount为负,0,10996,0.0000,无需处理,检出0条
5,订单discount_amount为负,0,10996,0.0000,无需处理,检出0条
6,支付时间早于下单时间,0,10996,0.0000,无需处理,检出0条
7,明细quantity<=0,0,27523,0.0000,无需处理,检出0条（数量恒为1）
8,明细item_total为负,0,27523,0.0000,无需处理,检出0条
9,item_total不等于数量×单价,0,27523,0.0000,无需处理,检出0条，27523行全部精确勾稽


## 模块 6｜数据质量校验（五）：字典编码合法性与口径性结构检查
本格目标：校验业务表编码是否都能命中字典，并记录影响口径的结构性现象（多余字典码、配送方式重名、支付时间全量非空、库存/设备单值等）。

In [7]:
# 平台编码：order/user/product/behavior 四表对 platform.code
illegal_platform = sum(int((~raw[t]["platform"].isin(set(platform.code))).sum())
                       for t in ["user", "product", "user_behavior", "order"])
# 状态/支付/配送存中文名，对字典 name
illegal_status = int((~order.order_status.isin(set(order_status.name))).sum())
illegal_pay = int((~order.payment_method.isin(set(payment_method.name))).sum())
illegal_ship = int((~order.shipping_method.isin(set(shipping_method.name))).sum())

dq_dict = pd.DataFrame([
    {"校验项": "平台编码非法条数(四表合计)", "结果": illegal_platform},
    {"校验项": "订单状态中文名非法条数", "结果": illegal_status},
    {"校验项": "支付方式中文名非法条数", "结果": illegal_pay},
    {"校验项": "配送方式中文名非法条数", "结果": illegal_ship},
    {"校验项": "platform字典中无业务数据的编码", "结果": "pinduoduo, tiktok_shop（保留字典不删除，统计限定三平台）"},
    {"校验项": "shipping_method重名记录", "结果": str(shipping_method[shipping_method.name == "同城配送"]["code"].tolist()) + " 同名，JOIN前按name去重"},
    {"校验项": "待付款/已取消订单仍有payment_time",
     "结果": f"{int((order.order_status.isin(['待付款','已取消'])).sum())}笔；判定支付以行为表「支付」事件和order_status为准，不以payment_time非空为准"},
    {"校验项": "stock_status取值", "结果": str(product.stock_status.value_counts().to_dict())},
    {"校验项": "device_type取值", "结果": str(user_behavior.device_type.value_counts().to_dict()) + "（单值，不做设备对比结论）"},
    {"校验项": "total_amount与明细合计差异",
     "结果": "10996/10996订单恒等，优惠/运费/税费为独立字段未参与扣减，GMV直接取total_amount"},
])
dq_dict.to_csv(MET / "dq_dict.csv", index=False, encoding="utf-8-sig")
dq_dict

,校验项,结果
0,平台编码非法条数(四表合计),0
1,订单状态中文名非法条数,0
2,支付方式中文名非法条数,0
3,配送方式中文名非法条数,0
4,platform字典中无业务数据的编码,"pinduoduo, tiktok_shop（保留字典不删除，统计限定三平台）"
5,shipping_method重名记录,"['local', 'same_city'] 同名，JOIN前按name去重"
6,待付款/已取消订单仍有payment_time,4401笔；判定支付以行为表「支付」事件和order_status为准，不以payment_...
7,stock_status取值,{'有货': 3298}
8,device_type取值,{'mobile': 21998}（单值，不做设备对比结论）
9,total_amount与明细合计差异,10996/10996订单恒等，优惠/运费/税费为独立字段未参与扣减，GMV直接取total...


## 模块 7｜字典映射（pandas merge 备选方案）

本格目标：用 pandas merge 完成订单主表与四张字典表、用户表的 LEFT JOIN，作为 SQL 关联的备选实现与交叉验证；断言关联后行数不放大（重点防御「同城配送」重名翻倍）。

In [8]:
# 配送方式按中文名去重（与 SQL 的 MIN(code) 口径一致）
ship_dedup = shipping_method.groupby("name", as_index=False).code.min()
pay_dedup = payment_method.groupby("name", as_index=False).code.min()

order_enriched_pd = (
    order
    .merge(platform[["code", "name"]].rename(columns={"name": "platform_name"}),
           how="left", left_on="platform", right_on="code")
    .merge(order_status[["code", "name"]].rename(columns={"code": "status_code", "name": "status_name"}),
           how="left", left_on="order_status", right_on="status_name")
    .merge(pay_dedup.rename(columns={"code": "payment_method_code", "name": "payment_method_name"}),
           how="left", left_on="payment_method", right_on="payment_method_name")
    .merge(ship_dedup.rename(columns={"code": "shipping_method_code", "name": "shipping_method_name"}),
           how="left", left_on="shipping_method", right_on="shipping_method_name")
    .merge(user[["global_user_id", "gender", "age", "city", "user_level", "registration_date"]],
           how="left", on="global_user_id")
)
# 关键断言：字典 LEFT JOIN 后行数必须与原订单表完全一致（若重名未去重会在此失败）
assert len(order_enriched_pd) == len(order) == 10996, "字典关联导致行数放大！"
print("pandas 备选关联通过，订单宽表行数：", len(order_enriched_pd))
order_enriched_pd[["order_id", "platform", "platform_name", "order_status", "status_code",
                   "payment_method", "shipping_method"]].head(3)

pandas 备选关联通过，订单宽表行数： 10996


,order_id,platform,platform_name,order_status,status_code,payment_method,shipping_method
0,DOUYIN20250101732638,douyin,抖音,已付款,paid,微信支付,快递配送
1,DOUYIN20250101858401,douyin,抖音,已取消,cancelled,银行卡,同城配送
2,DOUYIN20250101883803,douyin,抖音,已完成,completed,银行卡,同城配送


## 模块 8｜执行 `code/data_process.sql`（核心指标的 SQL 主实现）

本格目标：把 9 张 CSV 载入内存 SQLite，解析 SQL 脚本中的 `-- [[SECTION: 名称]]` 分段，先建宽表/RFM 临时视图（setup），再逐段执行 16 个指标/质量查询，结果存入字典 `S` 并落盘 `output/metrics/`。核心业务指标一律以 SQL 为准。

In [9]:
con = sqlite3.connect(":memory:")
for n in BUSINESS_FILES:
    raw[n].to_sql(n, con, index=False, if_exists="replace")

# 解析 data_process.sql：去块注释 -> 按 SECTION 标记切分 -> 去整行注释
sql_text = (CODE / "data_process.sql").read_text(encoding="utf-8")
sql_nc = re.sub(r"/\*.*?\*/", "", sql_text, flags=re.S)
parts = re.split(r"-- \[\[SECTION:\s*([a-z0-9_]+)\]\]", sql_nc)
sections = {}
for i in range(1, len(parts), 2):
    body = "\n".join(ln for ln in parts[i + 1].splitlines() if not re.match(r"\s*--", ln)).strip()
    sections[parts[i].strip()] = body

con.executescript(sections.pop("setup"))   # 建立 v_order_enriched / v_item_wide / v_behavior_wide / v_rfm 等视图
S = {name: pd.read_sql_query(q, con) for name, q in sections.items()}
for name, df in S.items():                  # 每段结果都保存为中间结果 CSV
    df.to_csv(MET / f"{name}.csv", index=False, encoding="utf-8-sig")
con.close()
print("SQL 分段执行完成：", len(S), "个结果表 ->", list(S.keys()))

SQL 分段执行完成： 21 个结果表 -> ['dq_fk', 'dq_business', 'dq_item', 'dq_session', 'pareto_cumulative', 'dq_dict', 'core_metrics', 'funnel_overall', 'funnel_platform', 'level_metrics', 'category_sales', 'monthly_gmv', 'cohort_retention', 'rfm_summary', 'rfm_detail', 'pareto_user', 'pareto_sku', 'product_label', 'product_label_summary', 'category_funnel', 'platform_level_compare']


## 模块 9｜指标①：全网 + 分平台 GMV / 有效订单 / 购买用户 / 客单价（SQL core_metrics）

In [10]:
S["core_metrics"]

,platform_code,platform_name,gmv,valid_orders,buyer_users,aov
0,ALL,全网,"5,692,092.1300",2216,1643,"2,568.6300"
1,taobao,淘宝,"2,322,985.6300",912,630,"2,547.1300"
2,douyin,抖音,"1,752,679.6400",673,569,"2,604.2800"
3,jd,京东,"1,616,426.8600",631,444,"2,561.6900"


## 模块 10｜指标②：浏览→加购→下单→支付漏斗（整体 + 分平台）
口径：各环节去重用户数；环节转化率用 SQL LAG()，累计转化率用 FIRST_VALUE()。

In [11]:
display(S["funnel_overall"])
S["funnel_platform"]

,stage,user_cnt,stage_conv_pct,total_conv_pct
0,浏览,2711,NaN,100.0000
1,加购,2368,87.3500,87.3500
2,下单,1953,82.4700,72.0400
3,支付,1734,88.7900,63.9600


,platform_code,platform_name,stage,user_cnt,stage_conv_pct,total_conv_pct
0,douyin,抖音,浏览,1124,NaN,100.0000
1,douyin,抖音,加购,918,81.6700,81.6700
2,douyin,抖音,下单,670,72.9800,59.6100
3,douyin,抖音,支付,579,86.4200,51.5100
4,jd,京东,浏览,742,NaN,100.0000
5,jd,京东,加购,703,94.7400,94.7400
6,jd,京东,下单,661,94.0300,89.0800
7,jd,京东,支付,617,93.3400,83.1500
8,taobao,淘宝,浏览,845,NaN,100.0000
9,taobao,淘宝,加购,747,88.4000,88.4000


## 模块 11｜指标③：会员等级订单量与 GMV 贡献（SQL level_metrics）

In [12]:
S["level_metrics"]

,user_level,valid_orders,buyer_users,gmv,gmv_pct,order_pct,aov
0,钻石会员,604,427,"1,570,928.8200",27.6000,27.2600,"2,600.8800"
1,普通会员,559,416,"1,451,632.3400",25.5000,25.2300,"2,596.8400"
2,银卡会员,539,423,"1,372,368.8200",24.1100,24.3200,"2,546.1400"
3,金卡会员,514,377,"1,297,162.1500",22.7900,23.1900,"2,523.6600"


## 模块 12｜指标④：品类销售、品类转化与月度 GMV 趋势

In [13]:
display(S["category_sales"])
display(S["category_funnel"])
S["monthly_gmv"].head(6)

,category,gmv,valid_orders,sold_sku_cnt
0,食品饮料,"777,447.6100",694,269
1,家居用品,"752,407.1300",660,252
2,图书文具,"745,180.0200",662,266
3,母婴用品,"740,566.1000",649,264
4,电子产品,"684,473.0700",624,247
5,服装鞋帽,"676,484.2700",607,239
6,运动户外,"670,709.7700",593,256
7,美妆个护,"644,824.1600",579,250


,category,browse_users,order_users,browse_to_order_pct
0,运动户外,546,164,30.0400
1,食品饮料,617,174,28.2000
2,母婴用品,692,182,26.3000
3,美妆个护,578,151,26.1200
4,图书文具,601,151,25.1200
5,家居用品,541,133,24.5800
6,电子产品,529,120,22.6800
7,服装鞋帽,468,98,20.9400


,month_id,gmv,valid_orders,buyer_users,aov
0,2025-01,"249,121.4900",97,97,"2,568.2600"
1,2025-02,"248,527.1700",95,93,"2,616.0800"
2,2025-03,"227,858.5000",91,89,"2,503.9400"
3,2025-04,"220,768.5700",96,95,"2,299.6700"
4,2025-05,"227,490.6500",83,83,"2,740.8500"
5,2025-06,"219,146.8200",87,82,"2,518.9300"


## 模块 13｜SQL ↔ Python 交叉校验（三方校验之 SQL↔Python 两条腿；Excel 腿见根目录 cross_check.xlsx）
本格目标：pandas 独立重算核心指标（不引用 SQL 结果），逐项比对并落盘 `cross_check_sql_python.csv`；Excel 腿按根目录 `excel_cross_check.md` 人工复算。

In [14]:
completed = order[order.order_status == "已完成"]
fun_py = (user_behavior[user_behavior.behavior_type.isin(["浏览", "加购", "下单", "支付"])]
          .groupby("behavior_type").global_user_id.nunique())
cat_gmv_py = (order_item.merge(completed[["order_id"]], on="order_id")
              .merge(product[["global_product_id", "category"]], on="global_product_id")
              .groupby("category").item_total.sum())

cross = pd.DataFrame([
    ["全网GMV(元)", S["core_metrics"].iloc[0].gmv, round(completed.total_amount.sum(), 2)],
    ["有效订单量", S["core_metrics"].iloc[0].valid_orders, completed.order_id.nunique()],
    ["有效购买用户数", S["core_metrics"].iloc[0].buyer_users, completed.global_user_id.nunique()],
    ["客单价(元)", S["core_metrics"].iloc[0].aov,
     round(completed.total_amount.sum() / completed.order_id.nunique(), 2)],
    ["漏斗-浏览用户", S["funnel_overall"].iloc[0].user_cnt, fun_py["浏览"]],
    ["漏斗-加购用户", S["funnel_overall"].iloc[1].user_cnt, fun_py["加购"]],
    ["漏斗-下单用户", S["funnel_overall"].iloc[2].user_cnt, fun_py["下单"]],
    ["漏斗-支付用户", S["funnel_overall"].iloc[3].user_cnt, fun_py["支付"]],
    ["食品饮料品类GMV(元)", S["category_sales"].query("category=='食品饮料'").gmv.iloc[0],
     round(cat_gmv_py["食品饮料"], 2)],
], columns=["指标", "SQL结果", "Python结果"])
cross["差异绝对值"] = (cross["SQL结果"] - cross["Python结果"]).abs().round(2)
cross["是否一致"] = np.where(cross["差异绝对值"] < 0.05, "一致", "不一致")
cross.to_csv(MET / "cross_check_sql_python.csv", index=False, encoding="utf-8-sig")
cross

,指标,SQL结果,Python结果,差异绝对值,是否一致
0,全网GMV(元),"5,692,092.1300","5,692,092.1300",0.0000,一致
1,有效订单量,"2,216.0000","2,216.0000",0.0000,一致
2,有效购买用户数,"1,643.0000","1,643.0000",0.0000,一致
3,客单价(元),"2,568.6300","2,568.6300",0.0000,一致
4,漏斗-浏览用户,"2,711.0000","2,711.0000",0.0000,一致
5,漏斗-加购用户,"2,368.0000","2,368.0000",0.0000,一致
6,漏斗-下单用户,"1,953.0000","1,953.0000",0.0000,一致
7,漏斗-支付用户,"1,734.0000","1,734.0000",0.0000,一致
8,食品饮料品类GMV(元),"777,447.6100","777,447.6100",0.0000,一致


## 模块 14｜阶段2-① Cohort 30/60/90 天留存（SQL cohort_retention）

口径：注册月份分批次，注册后第 1~N 天内有已完成订单计为留存；`complete_*` 列标识窗口是否在快照日前走完。注册前订单（负 offset）不计入。

In [15]:
cohort = S["cohort_retention"]
complete_mean = cohort[cohort.complete_90d == 1][
    ["retention_30d_pct", "retention_60d_pct", "retention_90d_pct"]].mean().round(2)
print("窗口完整批次平均留存率：", complete_mean.to_dict())
cohort.head(8)

窗口完整批次平均留存率： {'retention_30d_pct': 1.78, 'retention_60d_pct': 3.22, 'retention_90d_pct': 4.66}


,cohort_month,cohort_size,retained_30d,retained_60d,retained_90d,retention_30d_pct,retention_60d_pct,retention_90d_pct,complete_30d,complete_60d,complete_90d
0,2025-01,198,1,6,10,0.5100,3.0300,5.0500,1,1,1
1,2025-02,241,6,8,10,2.4900,3.3200,4.1500,1,1,1
2,2025-03,241,4,6,7,1.6600,2.4900,2.9000,1,1,1
3,2025-04,219,4,6,8,1.8300,2.7400,3.6500,1,1,1
4,2025-05,205,1,5,12,0.4900,2.4400,5.8500,1,1,1
5,2025-06,230,6,9,14,2.6100,3.9100,6.0900,1,1,1
6,2025-07,239,4,6,9,1.6700,2.5100,3.7700,1,1,1
7,2025-08,211,1,4,4,0.4700,1.9000,1.9000,1,1,1


## 模块 15｜阶段2-② RFM 分层（SQL NTILE(4) 打分 + CASE 规则）

规则：R/F/M 均≥3 高价值；R≤2 且（F≥3 或 M≥3）流失预警；R≥3 其余潜力；剩余低价值。仅覆盖 1,643 名有效消费用户。

In [18]:
display(S["rfm_summary"])
print("无有效订单的注册用户数：",
      user.global_user_id.nunique() - S["rfm_detail"].global_user_id.nunique())

,segment,users,user_pct,gmv,gmv_pct,avg_r_days,avg_f_cnt,avg_m_amt
0,高价值用户,434,26.4200,"2,645,824.2700",46.4800,136.5000,1.9700,"6,096.3700"
1,流失预警用户,380,23.1300,"1,708,110.6600",30.0100,504.5000,1.3000,"4,495.0300"
2,低价值用户,442,26.9000,"696,317.1500",12.2300,517.1000,1.0000,"1,575.3800"
3,潜力用户,387,23.5500,"641,840.0500",11.2800,150.4000,1.0900,"1,658.5000"


无有效订单的注册用户数： 3843


## 模块 16｜阶段2-③ 帕累托：头部 20% 用户 / SKU 的 GMV 集中度（SQL NTILE(5)）

In [19]:
display(S["pareto_user"])
S["pareto_sku"]

,top_user_cnt,top20_gmv_pct,total_gmv
0,329,43.4000,"5,692,092.1300"


,top_sku_cnt,top20_gmv_pct,total_gmv
0,409,47.6300,"5,692,092.1300"


## 模块 17｜阶段2-④ 商品分层：爆品 / 动销 / 滞销（SQL product_label 系列）
滞销=零有效成交；爆品=动销 SKU 中 GMV 前 20% 且有货。另统计从未被下单的 SKU。

In [20]:
display(S["product_label_summary"])
sold_any = set(order_item.global_product_id)
zero_ever = int((~product.global_product_id.isin(sold_any)).sum())
print(f"零有效成交SKU {int(S['product_label_summary'].set_index('sku_label').loc['滞销SKU','sku_cnt'])} 个；"
      f"其中全窗口从未被下单 {zero_ever} 个，仅出现在非完成状态订单 "
      f"{int(S['product_label_summary'].set_index('sku_label').loc['滞销SKU','sku_cnt']) - zero_ever} 个")
S["product_label"].head(5)

,sku_label,sku_cnt,sku_pct,gmv,gmv_pct
0,动销SKU,1634,49.5500,"2,980,981.4800",52.3700
1,爆品SKU,409,12.4000,"2,711,110.6500",47.6300
2,滞销SKU,1255,38.0500,0.0000,0.0000


零有效成交SKU 1255 个；其中全窗口从未被下单 358 个，仅出现在非完成状态订单 897 个


,global_product_id,category,brand,list_price,stock_status,sales_qty,sales_gmv,sku_label
0,P-DOUYIN-675543,服装鞋帽,耐克,676.0100,有货,9,"15,167.2000",爆品SKU
1,P-DOUYIN-696100,食品饮料,优衣库,"3,159.2300",有货,10,"12,838.6300",爆品SKU
2,P-TAOBAO-171982,运动户外,Apple,"1,375.1800",有货,8,"12,500.6000",爆品SKU
3,P-TAOBAO-910500,家居用品,Apple,"3,586.6900",有货,10,"12,291.6000",爆品SKU
4,P-TAOBAO-699532,母婴用品,格力,"1,443.1300",有货,12,"12,167.2900",爆品SKU


## 模块 18｜阶段2-⑤ 平台 × 会员等级多维对比（注册用户→有效购买转化率）

In [21]:
pl_compare = S["platform_level_compare"]
display(pl_compare)
# 平台级汇总购买转化率
pl_compare.groupby("platform_code")[["register_users", "buyer_users"]].sum().assign(
    buyer_conv_pct=lambda d: (100 * d.buyer_users / d.register_users).round(2))

,platform_code,user_level,register_users,buyer_users,buyer_conv_pct,valid_orders,gmv,aov
0,douyin,银卡会员,649,156,24.0400,175,"467,072.5900","2,668.9900"
1,douyin,钻石会员,625,143,22.8800,171,"449,436.3800","2,628.2800"
2,douyin,普通会员,609,148,24.3000,173,"445,033.5200","2,572.4500"
3,douyin,金卡会员,584,122,20.8900,154,"391,137.1500","2,539.8500"
4,jd,钻石会员,412,134,32.5200,213,"534,113.8300","2,507.5800"
5,jd,普通会员,374,106,28.3400,145,"392,586.8600","2,707.5000"
6,jd,银卡会员,406,107,26.3500,145,"357,650.4900","2,466.5600"
7,jd,金卡会员,359,97,27.0200,128,"332,075.6800","2,594.3400"
8,taobao,普通会员,381,162,42.5200,241,"614,011.9600","2,547.7700"
9,taobao,钻石会员,343,150,43.7300,220,"587,378.6100","2,669.9000"


,register_users,buyer_users,buyer_conv_pct
platform_code,,,
douyin,2467,569,23.0600
jd,1551,444,28.6300
taobao,1468,630,42.9200


## 模块 19｜阶段3 EDA（一）：年龄段 / 城市的转化率与客单价分布

In [22]:
buyer = completed[["global_user_id", "order_id", "total_amount"]].merge(user, on="global_user_id")
age_bins, age_labels = [17, 25, 35, 45, 60], ["18-25岁", "26-35岁", "36-45岁", "46-60岁"]
user["age_band"] = pd.cut(user.age, bins=age_bins, labels=age_labels)
buyer["age_band"] = pd.cut(buyer.age, bins=age_bins, labels=age_labels)

def dim_table(dim):
    reg = user.groupby(dim, observed=True).global_user_id.nunique().rename("注册用户数")
    buy = buyer.groupby(dim, observed=True).global_user_id.nunique().rename("有效购买用户数")
    odn = buyer.groupby(dim, observed=True).order_id.nunique().rename("有效订单量")
    gm = buyer.groupby(dim, observed=True).total_amount.sum().round(2).rename("GMV")
    t = pd.concat([reg, buy, odn, gm], axis=1).fillna(0)
    t["购买转化率_%"] = (100 * t["有效购买用户数"] / t["注册用户数"]).round(2)
    t["客单价_元"] = (t["GMV"] / t["有效订单量"]).round(2)
    return t.reset_index()

eda_age = dim_table("age_band")
eda_city = dim_table("city").sort_values("购买转化率_%", ascending=False)
eda_age.to_csv(MET / "eda_age.csv", index=False, encoding="utf-8-sig")
eda_city.to_csv(MET / "eda_city.csv", index=False, encoding="utf-8-sig")
display(eda_age)
eda_city

,age_band,注册用户数,有效购买用户数,有效订单量,GMV,购买转化率_%,客单价_元
0,18-25岁,1011,309,407,"1,072,600.2600",30.5600,"2,635.3800"
1,26-35岁,1258,350,461,"1,228,384.7100",27.8200,"2,664.6100"
2,36-45岁,1326,389,514,"1,287,950.1100",29.3400,"2,505.7400"
3,46-60岁,1891,595,834,"2,103,157.0500",31.4600,"2,521.7700"


,city,注册用户数,有效购买用户数,有效订单量,GMV,购买转化率_%,客单价_元
7,深圳,517,166,242,"652,114.1200",32.1100,"2,694.6900"
4,成都,550,174,227,"561,548.0900",31.6400,"2,473.7800"
8,西安,562,175,225,"554,622.9900",31.1400,"2,464.9900"
6,武汉,538,166,222,"558,612.1400",30.8600,"2,516.2700"
1,北京,562,171,241,"606,087.1900",30.4300,"2,514.8800"
2,南京,552,163,206,"542,569.4100",29.5300,"2,633.8300"
5,杭州,573,166,215,"561,403.1100",28.9700,"2,611.1800"
3,广州,561,161,201,"522,368.0700",28.7000,"2,598.8500"
9,重庆,547,156,253,"653,377.3200",28.5200,"2,582.5200"
0,上海,524,145,184,"479,389.6900",27.6700,"2,605.3800"


## 模块 20｜阶段3 EDA（二）：商品价格带转化与 GMV 结构；设备类型可行性检查

In [23]:
pbins = [0, 200, 800, 2000, 4000, 5000]
plabels = ["0-200元", "200-800元", "800-2000元", "2000-4000元", "4000-5000元"]
product["price_band"] = pd.cut(product.price, bins=pbins, labels=plabels)
ubp = user_behavior.merge(product[["global_product_id", "price_band"]], on="global_product_id")
br_n = (ubp[ubp.behavior_type == "浏览"].drop_duplicates(["global_user_id", "price_band"])
        .groupby("price_band", observed=True).size().rename("浏览用户数"))
od_n = (ubp[ubp.behavior_type == "下单"].drop_duplicates(["global_user_id", "price_band"])
        .groupby("price_band", observed=True).size().rename("下单用户数"))
itw = (order_item.merge(completed[["order_id"]], on="order_id")
       .merge(product[["global_product_id", "price_band"]], on="global_product_id"))
gmv_n = itw.groupby("price_band", observed=True).item_total.sum().round(2).rename("GMV")
eda_price = pd.concat([br_n, od_n, gmv_n], axis=1).fillna(0)
eda_price["浏览下单转化率_%"] = (100 * eda_price["下单用户数"] / eda_price["浏览用户数"]).round(2)
eda_price["GMV占比_%"] = (100 * eda_price["GMV"] / eda_price["GMV"].sum()).round(2)
eda_price = eda_price.reset_index()
eda_price.to_csv(MET / "eda_price_band.csv", index=False, encoding="utf-8-sig")
display(eda_price)

# 设备类型可行性检查：单值维度不允许产出对比结论
print("device_type 取值分布：", user_behavior.device_type.value_counts(dropna=False).to_dict())

,price_band,浏览用户数,下单用户数,GMV,浏览下单转化率_%,GMV占比_%
0,0-200元,157,93,"217,700.5100",59.2400,3.8200
1,200-800元,490,306,"790,382.3100",62.4500,13.8900
2,800-2000元,955,606,"1,382,929.2000",63.4600,24.3000
3,2000-4000元,1599,978,"2,228,457.1000",61.1600,39.1500
4,4000-5000元,822,491,"1,072,623.0100",59.7300,18.8400


device_type 取值分布： {'mobile': 21998}


## 模块 21｜阶段3 EDA（三）：RFM 四类用户的订单间隔与客单价差异

In [24]:
csort = completed.sort_values(["global_user_id", "order_time"]).copy()
csort["gap_days"] = csort.groupby("global_user_id").order_time.diff().dt.days
interval = csort.groupby("global_user_id").gap_days.mean().rename("avg_interval")
rfm_detail = S["rfm_detail"].merge(interval, on="global_user_id", how="left")
rfm_beh = rfm_detail.groupby("segment").agg(
    用户数=("global_user_id", "count"),
    平均订单间隔_天=("avg_interval", "mean"),
    平均消费频次=("f_cnt", "mean"),
    人均累计消费_元=("m_amt", "mean")).round(2).reset_index()
rfm_beh["平均客单价_元"] = (rfm_beh["人均累计消费_元"] / rfm_beh["平均消费频次"]).round(2)
rfm_beh.to_csv(MET / "eda_rfm_behavior.csv", index=False, encoding="utf-8-sig")
rfm_beh

,segment,用户数,平均订单间隔_天,平均消费频次,人均累计消费_元,平均客单价_元
0,低价值用户,442,NaN,1.0000,"1,575.3800","1,575.3800"
1,流失预警用户,380,135.7300,1.3000,"4,495.0300","3,457.7200"
2,潜力用户,387,268.4500,1.0900,"1,658.5000","1,521.5600"
3,高价值用户,434,253.9600,1.9700,"6,096.3700","3,094.6000"


## 模块 21B｜附录增强计算：LTV/ARPU · Cohort 口径敏感性 · 严格路径漏斗

本模块为评审增强项，只**新增**结果文件（`ltv_segment.csv`、`ltv_reg_month.csv`、`cohort_sensitivity.csv`、`funnel_strict_path.csv`），不改写任何既有指标 CSV，原报告结论与数字保持不变；两组敏感性口径并列呈现，供口径对照。

In [25]:
# ============================================================
# 模块 21B｜附录增强计算：LTV/ARPU、Cohort 口径敏感性、严格路径漏斗
# 原则：只新增结果文件，不改写既有指标；新增数字全部由 9 张原始 CSV 实算
# ============================================================

# ---------- 1) LTV / ARPU：两种分母口径（有购买用户 1,643 人 / 注册用户 5,486 人） ----------
rfm_detail = S["rfm_detail"]
N_BUYER = rfm_detail.global_user_id.nunique()    # 有效购买用户 1643
N_REG = user.global_user_id.nunique()            # 全部注册用户 5486
total_gmv = rfm_detail.m_amt.sum()               # 5,692,092.13，与 core_metrics 勾稽

# 1.1 按 RFM 分层：人均累计消费（分母=该层有购买用户，即 M 均值）+ 摊到全量注册用户的参考值
ltv_segment = (rfm_detail.groupby("segment")
               .agg(users=("global_user_id", "nunique"),
                    gmv=("m_amt", "sum"),
                    ltv_per_buyer=("m_amt", "mean"))
               .reset_index())
ltv_segment["gmv"] = ltv_segment["gmv"].round(2)
ltv_segment["ltv_per_buyer"] = ltv_segment["ltv_per_buyer"].round(2)
ltv_segment["gmv_per_registered"] = (ltv_segment["gmv"] / N_REG).round(2)

# 1.2 两个全网人均口径
ltv_overview = pd.DataFrame([
    {"指标": "有效购买用户人均累计消费 LTV（元/购买用户）", "分母人数": N_BUYER,
     "数值": round(total_gmv / N_BUYER, 2)},
    {"指标": "注册用户人均 GMV ARPU（元/注册用户）", "分母人数": N_REG,
     "数值": round(total_gmv / N_REG, 2)},
])

# 1.3 按注册月份的人均 GMV 趋势（两种分母并列，分母口径见列名）
_reg = user.copy()
_reg["reg_month"] = pd.to_datetime(_reg["registration_date"]).dt.strftime("%Y-%m")
_reg_users = _reg.groupby("reg_month").global_user_id.nunique().rename("reg_users")
_buy = (completed.groupby("global_user_id").total_amount.sum().rename("gmv").reset_index()
        .merge(_reg[["global_user_id", "reg_month"]], on="global_user_id", how="left"))
ltv_reg_month = (_buy.groupby("reg_month")
                 .agg(buyer_users=("global_user_id", "nunique"), gmv=("gmv", "sum"))
                 .join(_reg_users, how="right").fillna(0).reset_index())
ltv_reg_month["ltv_per_buyer"] = (ltv_reg_month.gmv /
                                  ltv_reg_month.buyer_users.replace(0, np.nan)).round(2)
ltv_reg_month["arpu_per_registered"] = (ltv_reg_month.gmv / ltv_reg_month.reg_users).round(2)

ltv_segment.to_csv(MET / "ltv_segment.csv", index=False, encoding="utf-8-sig")
ltv_reg_month.to_csv(MET / "ltv_reg_month.csv", index=False, encoding="utf-8-sig")
print(ltv_overview.to_string(index=False))
print(ltv_segment.to_string(index=False))
print(f"按注册月份：注册口径 ARPU 区间 {ltv_reg_month.arpu_per_registered.min()}~"
      f"{ltv_reg_month.arpu_per_registered.max()} 元；"
      f"购买用户人均 LTV 区间 {ltv_reg_month.ltv_per_buyer.min()}~{ltv_reg_month.ltv_per_buyer.max()} 元")

# ---------- 2) Cohort 留存敏感性：注册前订单全部计入（宽松口径 offset <= N） ----------
# 原口径（严格）：仅计 offset BETWEEN 1 AND N；宽松口径：注册前/当日订单全部计入 offset <= N
_regmap = pd.to_datetime(user.set_index("global_user_id").registration_date)
_co = completed[["global_user_id", "order_time"]].copy()
_co["order_time"] = pd.to_datetime(_co["order_time"])
_co["reg"] = _co.global_user_id.map(_regmap)
_co["off_days"] = (_co.order_time - _co.reg).dt.days
_uh = (_co.groupby("global_user_id")
       .agg(h30=("off_days", lambda s: int((s <= 30).any())),
            h60=("off_days", lambda s: int((s <= 60).any())),
            h90=("off_days", lambda s: int((s <= 90).any())))
       .reset_index())
_uh["cohort_month"] = _uh.global_user_id.map(_regmap.dt.strftime("%Y-%m"))
_lret = (_uh.groupby("cohort_month")[["h30", "h60", "h90"]].sum()
         .rename(columns={"h30": "lenient_retained_30d", "h60": "lenient_retained_60d",
                          "h90": "lenient_retained_90d"}).reset_index())
cohort_sensitivity = (
    S["cohort_retention"][["cohort_month", "cohort_size",
                           "retention_30d_pct", "retention_60d_pct", "retention_90d_pct",
                           "complete_30d", "complete_60d", "complete_90d"]]
    .merge(_lret, on="cohort_month", how="left").fillna(0))
cohort_sensitivity = cohort_sensitivity.rename(columns={
    "retention_30d_pct": "strict_retention_30d", "retention_60d_pct": "strict_retention_60d",
    "retention_90d_pct": "strict_retention_90d"})
for _n in (30, 60, 90):
    cohort_sensitivity[f"lenient_retention_{_n}d"] = (
        100 * cohort_sensitivity[f"lenient_retained_{_n}d"] / cohort_sensitivity.cohort_size).round(2)
cohort_sensitivity.to_csv(MET / "cohort_sensitivity.csv", index=False, encoding="utf-8-sig")
# 与主口径一致：三个窗口均取 90 天窗口完整的 21 个批次平均
_ok = cohort_sensitivity.complete_90d == 1
print("严格口径（注册前订单不计入）21 批均值：",
      cohort_sensitivity[_ok][["strict_retention_30d", "strict_retention_60d",
                               "strict_retention_90d"]].mean().round(2).to_dict())
print("宽松口径（注册前订单全计入）21 批均值：",
      cohort_sensitivity[_ok][["lenient_retention_30d", "lenient_retention_60d",
                               "lenient_retention_90d"]].mean().round(2).to_dict())

# ---------- 3) 严格路径漏斗：加购须先有浏览、下单须有浏览+加购、支付须前三环节齐全 ----------
# 主口径是「环节独立去重用户」；此处按用户×平台构造嵌套集合，628 名「有加购无浏览」用户将从漏斗消失
_ff = (user_behavior.groupby(["platform", "global_user_id", "behavior_type"]).size()
       .unstack(fill_value=0).reset_index())
for _s in ["浏览", "加购", "下单", "支付"]:
    if _s not in _ff.columns:
        _ff[_s] = 0
_ff["b"], _ff["c"] = _ff["浏览"] > 0, _ff["加购"] > 0
_ff["o"], _ff["pay"] = _ff["下单"] > 0, _ff["支付"] > 0
_rows, _PNAME = [], {"taobao": "淘宝", "jd": "京东", "douyin": "抖音"}
for _pc, _d in _ff.groupby("platform"):
    _b = int(_d.b.sum())
    _c = int((_d.b & _d.c).sum())
    _o = int((_d.b & _d.c & _d.o).sum())
    _p = int((_d.b & _d.c & _d.o & _d.pay).sum())
    _rows.append((_PNAME.get(_pc, _pc), _b, _c, _o, _p))
_g = _ff.groupby("global_user_id")[["b", "c", "o", "pay"]].any()
_rows.append(("全网", int(_g.b.sum()), int((_g.b & _g.c).sum()),
              int((_g.b & _g.c & _g.o).sum()), int((_g.b & _g.c & _g.o & _g.pay).sum())))
funnel_strict_path = pd.DataFrame(
    _rows, columns=["platform", "browse_users", "cart_strict_users",
                    "order_strict_users", "pay_strict_users"])
funnel_strict_path["browse_to_cart_pct"] = (
    100 * funnel_strict_path.cart_strict_users / funnel_strict_path.browse_users).round(2)
funnel_strict_path["cart_to_order_pct"] = (
    100 * funnel_strict_path.order_strict_users / funnel_strict_path.cart_strict_users).round(2)
funnel_strict_path["order_to_pay_pct"] = (
    100 * funnel_strict_path.pay_strict_users / funnel_strict_path.order_strict_users).round(2)
funnel_strict_path["browse_to_pay_pct"] = (
    100 * funnel_strict_path.pay_strict_users / funnel_strict_path.browse_users).round(2)
funnel_strict_path.to_csv(MET / "funnel_strict_path.csv", index=False, encoding="utf-8-sig")
print(f"有加购无浏览记录用户（全网跨平台）：{int((_g.c & ~_g.b).sum())} 人；"
      f"有下单但无「浏览+加购」完整路径用户：{int((_g.o & ~(_g.b & _g.c)).sum())} 人")
funnel_strict_path

                      指标  分母人数         数值
有效购买用户人均累计消费 LTV（元/购买用户）  1643 3,464.4500
 注册用户人均 GMV ARPU（元/注册用户）  5486 1,037.5700
segment  users            gmv  ltv_per_buyer  gmv_per_registered
  低价值用户    442   696,317.1500     1,575.3800            126.9300
 流失预警用户    380 1,708,110.6600     4,495.0300            311.3600
   潜力用户    387   641,840.0500     1,658.5000            117.0000
  高价值用户    434 2,645,824.2700     6,096.3700            482.2900
按注册月份：注册口径 ARPU 区间 786.78~1434.59 元；购买用户人均 LTV 区间 2859.64~4367.05 元
严格口径（注册前订单不计入）21 批均值： {'strict_retention_30d': 1.78, 'strict_retention_60d': 3.22, 'strict_retention_90d': 4.66}
宽松口径（注册前订单全计入）21 批均值： {'lenient_retention_30d': 15.29, 'lenient_retention_60d': 16.49, 'lenient_retention_90d': 17.7}
有加购无浏览记录用户（全网跨平台）：628 人；有下单但无「浏览+加购」完整路径用户：948 人


,platform,browse_users,cart_strict_users,order_strict_users,pay_strict_users,browse_to_cart_pct,cart_to_order_pct,order_to_pay_pct,browse_to_pay_pct
0,抖音,1124,718,387,258,63.8800,53.9000,66.6700,22.9500
1,京东,742,454,262,176,61.1900,57.7100,67.1800,23.7200
2,淘宝,845,568,356,240,67.2200,62.6800,67.4200,28.4000
3,全网,2711,1740,1005,674,64.1800,57.7600,67.0600,24.8600


## 模块 22｜阶段3 轻量逻辑回归：下单转化的特征方向验证

使用 sklearn `LogisticRegression`（默认超参，不做网格搜索、不做模型性能优化）。目的不是训练上线模型，而是用系数方向（发生比 OR）验证「收藏/加购意图信号与下单最相关」，与漏斗、EDA 结论交叉印证；输出的训练集 AUC 仅描述样本内区分度，不代表泛化性能。

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# 样本：有浏览行为的用户；标签：是否有下单行为（与漏斗「浏览/下单去重用户」口径一致）
pivot_cnt = user_behavior.pivot_table(index="global_user_id", columns="behavior_type",
                                      values="behavior_id", aggfunc="count", fill_value=0).reset_index()
avg_dur = user_behavior.groupby("global_user_id").duration_seconds.mean().rename("avg_duration")
browse_set = set(user_behavior.loc[user_behavior.behavior_type == "浏览", "global_user_id"])
order_set = set(user_behavior.loc[user_behavior.behavior_type == "下单", "global_user_id"])
lrdf = (user[user.global_user_id.isin(browse_set)]
        .merge(pivot_cnt, on="global_user_id", how="left").merge(avg_dur, on="global_user_id", how="left"))
for c in ["浏览", "加购", "收藏", "avg_duration"]:
    lrdf[c] = lrdf[c].fillna(0) if c in lrdf.columns else 0
lrdf["level_ord"] = lrdf.user_level.map({"普通会员": 1, "银卡会员": 2, "金卡会员": 3, "钻石会员": 4})
lrdf["gender_m"] = (lrdf.gender == "m").astype(int)
lrdf["is_jd"] = (lrdf.platform == "jd").astype(int)
lrdf["is_douyin"] = (lrdf.platform == "douyin").astype(int)
lrdf["y"] = lrdf.global_user_id.isin(order_set).astype(int)

features = ["age", "gender_m", "level_ord", "浏览", "加购", "收藏", "avg_duration", "is_jd", "is_douyin"]
X = StandardScaler().fit_transform(lrdf[features].astype(float))
clf = LogisticRegression(max_iter=1000)   # 默认超参：仅做特征方向验证，不做网格搜索/性能优化
clf.fit(X, lrdf.y)
lr_coef = pd.DataFrame({"feature": features, "coef": clf.coef_[0].round(4)})
lr_coef["odds_ratio"] = np.exp(lr_coef.coef).round(3)
lr_coef = lr_coef.reindex(lr_coef.coef.abs().sort_values(ascending=False).index)
lr_coef.to_csv(MET / "lr_coef.csv", index=False, encoding="utf-8-sig")

# 样本内拟合指标单独落【新增】文件（不改写既有指标），仅用来说明模型具备基本区分度
train_auc = roc_auc_score(lrdf.y, clf.predict_proba(X)[:, 1])
train_acc = (clf.predict(X) == lrdf.y).mean()
lr_fit = pd.DataFrame([{
    "sample_n": len(lrdf), "pos_rate": round(lrdf.y.mean(), 4),
    "train_accuracy": round(train_acc, 4), "train_auc": round(train_auc, 4),
    "note": "训练集（全样本）样本内评估，仅作特征方向验证，非泛化性能指标、不用于预测上线"}])
lr_fit.to_csv(MET / "lr_model_fit.csv", index=False, encoding="utf-8-sig")
print(f"样本{len(lrdf)}人，正样本率{lrdf.y.mean():.2%}，样本内准确率{train_acc:.2%}，"
      f"训练集AUC={train_auc:.2f}（仅描述用）")
lr_coef

样本2711人，正样本率47.25%，样本内准确率71.49%，训练集AUC=0.78（仅描述用）


,feature,coef,odds_ratio
5,收藏,1.3642,3.9130
4,加购,0.8352,2.3050
0,age,-0.0619,0.9400
7,is_jd,0.0618,1.0640
8,is_douyin,-0.0521,0.9490
6,avg_duration,-0.0486,0.9530
3,浏览,-0.0276,0.9730
2,level_ord,-0.0135,0.9870
1,gender_m,-0.0091,0.9910


## 模块 23｜阶段4：可视化输出（fig01~fig08 分析图，保存至 figures/）

本单元输出前 8 张分析图（6 张规定图 + 2 张补充图）。**图 9~11 为 Power BI Desktop 中手工搭建的实物看板**，截图与源文件存放于根目录 `bi_screenshots/`（`bi_01~bi_03` + `.pbix`），搭建步骤见 `report/PowerBI搭建指南.md`。

In [27]:
COLORS = {"淘宝": "#FF6A00", "京东": "#E1251B", "抖音": "#161823"}
stages = ["浏览", "加购", "下单", "支付"]

# 图1：分平台转化漏斗对比（用户数分组柱 + 相邻环节转化率折线）
fp = S["funnel_platform"]
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
x = np.arange(4); w = 0.25
for i, (pc, pn) in enumerate(zip(["taobao", "jd", "douyin"], ["淘宝", "京东", "抖音"])):
    d = fp[fp.platform_code == pc].set_index("stage").reindex(stages)
    ax[0].bar(x + (i - 1) * w, d.user_cnt, width=w, label=pn, color=COLORS[pn])
    for xi, v in zip(x + (i - 1) * w, d.user_cnt):
        ax[0].text(xi, v + 8, str(int(v)), ha="center", fontsize=8)
ax[0].set_xticks(x); ax[0].set_xticklabels(stages)
ax[0].set_xlabel("漏斗环节"); ax[0].set_ylabel("去重用户数（人）")
ax[0].set_title("分平台漏斗各环节去重用户数"); ax[0].legend()
for i, (pc, pn) in enumerate(zip(["taobao", "jd", "douyin"], ["淘宝", "京东", "抖音"])):
    d = fp[fp.platform_code == pc].set_index("stage").reindex(stages)
    ax[1].plot(x[1:], d.stage_conv_pct.values[1:], marker="o", label=pn, color=COLORS[pn], linewidth=2)
    for xi, v in zip(x[1:], d.stage_conv_pct.values[1:]):
        ax[1].annotate(f"{v:.1f}%", (xi, v), textcoords="offset points", xytext=(0, 6),
                       ha="center", fontsize=8, color=COLORS[pn])
ax[1].set_xticks(x[1:]); ax[1].set_xticklabels(["浏览→加购", "加购→下单", "下单→支付"])
ax[1].set_ylabel("环节转化率（%）"); ax[1].set_ylim(60, 100)
ax[1].set_title("分平台相邻环节转化率对比"); ax[1].legend()
fig.suptitle("三大平台 浏览→加购→下单→支付 转化漏斗对比", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG / "fig01_分平台转化漏斗对比.png", dpi=150, bbox_inches="tight"); plt.show()

# 图2：Cohort 同期群留存热力图
mat = cohort.set_index("cohort_month")[["retention_30d_pct", "retention_60d_pct", "retention_90d_pct"]]
mat.columns = ["30天留存率", "60天留存率", "90天留存率"]
fig, ax = plt.subplots(figsize=(7.2, 9))
sns.heatmap(mat, annot=True, fmt=".2f", cmap="Blues", cbar_kws={"label": "留存率（%）"},
            linewidths=.4, ax=ax, annot_kws={"fontsize": 8})
ax.set_xlabel("注册后观察窗口"); ax.set_ylabel("注册批次（年-月）")
ax.set_title("用户 Cohort 同期群留存热力图（已完成订单口径）", fontsize=13, fontweight="bold")
plt.figtext(0.01, 0.01, "注：2026-10 起批次 60/90 天窗口未走完，数值为未完结窗口结果", fontsize=8, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1]); plt.savefig(FIG / "fig02_Cohort同期群留存热力图.png", dpi=150, bbox_inches="tight"); plt.show()

# 图3：RFM 四类用户人数占比环形图
rs = S["rfm_summary"]
fig, ax = plt.subplots(figsize=(7.6, 6))
wedges, _, _ = ax.pie(rs.users, labels=rs.segment,
                      autopct=lambda p: f"{p:.1f}%\n({int(round(p*rs.users.sum()/100))}人)",
                      colors=["#2E86AB", "#F4A259", "#5B8E7D", "#BC4749"], startangle=90,
                      pctdistance=0.72, wedgeprops=dict(width=0.42, edgecolor="white"))
ax.set_title("RFM 四类用户人数占比（有效消费用户 1643 人）", fontsize=13, fontweight="bold")
ax.legend(wedges, [f"{s}｜GMV贡献{g}%" for s, g in zip(rs.segment, rs.gmv_pct)],
          loc="lower center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "fig03_RFM用户分层占比饼图.png", dpi=150, bbox_inches="tight"); plt.show()

# 图4：各品类销售额柱状图
cs2 = S["category_sales"]
fig, ax = plt.subplots(figsize=(10, 5.2))
bars = ax.bar(cs2.category, cs2.gmv / 10000, color="#3A86FF", alpha=.85)
for b, v, n in zip(bars, cs2.gmv / 10000, cs2.valid_orders):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f"{v:.1f}万\n{n}单", ha="center", fontsize=8)
ax.set_ylabel("有效销售额（万元）"); ax.set_xlabel("一级品类")
ax.set_ylim(0, cs2.gmv.max() / 10000 * 1.18)
ax.set_title("各一级品类有效销售额与订单量（已完成订单）", fontsize=13, fontweight="bold")
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(FIG / "fig04_各品类销售额柱状图.png", dpi=150, bbox_inches="tight"); plt.show()

# 图5：爆品 & 滞销 SKU 数量结构与 GMV 贡献
pls = S["product_label_summary"].set_index("sku_label").reindex(["爆品SKU", "动销SKU", "滞销SKU"])
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
b1 = ax[0].bar(pls.index, pls.sku_cnt, color=["#E63946", "#F4A261", "#8D99AE"])
for b, v, p in zip(b1, pls.sku_cnt, pls.sku_pct):
    ax[0].text(b.get_x() + b.get_width() / 2, v + 12, f"{v}个\n{p}%", ha="center", fontsize=9)
ax[0].set_ylabel("SKU 数量（个）"); ax[0].set_title("爆品 / 动销 / 滞销 SKU 数量结构")
ax[0].set_ylim(0, pls.sku_cnt.max() * 1.18)
b2 = ax[1].bar(pls.index, pls.gmv_pct, color=["#E63946", "#F4A261", "#8D99AE"])
for b, v in zip(b2, pls.gmv_pct):
    ax[1].text(b.get_x() + b.get_width() / 2, v + 1, f"{v}%", ha="center", fontsize=9)
ax[1].set_ylabel("有效销售额贡献占比（%）"); ax[1].set_ylim(0, 60)
ax[1].set_title("三类 SKU 的有效销售额贡献占比")
fig.suptitle("商品分层：爆品与滞销 SKU 对比（共 3298 个 SKU）", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG / "fig05_爆品与滞销SKU对比.png", dpi=150, bbox_inches="tight"); plt.show()

# 图6：月度 GMV 趋势折线
mg = S["monthly_gmv"]
imax, imin = mg.gmv.idxmax(), mg.gmv.idxmin()
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(mg.month_id, mg.gmv / 10000, marker="o", color="#0077B6", linewidth=2)
ax.scatter(mg.month_id[imax], mg.gmv[imax] / 10000, color="#D62828", zorder=5, s=80)
ax.scatter(mg.month_id[imin], mg.gmv[imin] / 10000, color="#6A994E", zorder=5, s=80)
ax.annotate(f"峰值 {mg.gmv[imax]/10000:.1f}万", (mg.month_id[imax], mg.gmv[imax] / 10000),
            xytext=(0, 10), textcoords="offset points", ha="center", color="#D62828")
ax.annotate(f"谷值 {mg.gmv[imin]/10000:.1f}万", (mg.month_id[imin], mg.gmv[imin] / 10000),
            xytext=(0, -16), textcoords="offset points", ha="center", color="#6A994E")
ax.set_ylabel("月度有效 GMV（万元）"); ax.set_xlabel("月份")
ax.set_title("2025-01 至 2026-12 月度有效 GMV 趋势", fontsize=13, fontweight="bold")
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(FIG / "fig06_月度GMV趋势折线图.png", dpi=150, bbox_inches="tight"); plt.show()

# 图7（补充）：平台 × 会员等级购买转化率热力图
dl = S["platform_level_compare"]
piv = dl.pivot(index="user_level", columns="platform_code", values="buyer_conv_pct")[
    ["taobao", "jd", "douyin"]]
piv.columns = ["淘宝", "京东", "抖音"]
piv = piv.reindex(["普通会员", "银卡会员", "金卡会员", "钻石会员"])
fig, ax = plt.subplots(figsize=(7, 4.6))
sns.heatmap(piv, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={"label": "购买转化率（%）"}, ax=ax)
ax.set_xlabel("平台"); ax.set_ylabel("会员等级")
ax.set_title("分平台 × 会员等级 有效购买转化率（%）", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG / "fig07_平台会员等级转化率.png", dpi=150, bbox_inches="tight"); plt.show()

# 图8（补充）：年龄段与价格带转化率双面板
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.8))
ax[0].bar(eda_age.age_band.astype(str), eda_age["购买转化率_%"], color="#4361EE")
for i, v in enumerate(eda_age["购买转化率_%"]):
    ax[0].text(i, v + 0.4, f"{v}%", ha="center", fontsize=9)
ax[0].set_title("不同年龄段用户有效购买转化率"); ax[0].set_ylabel("转化率（%）"); ax[0].set_xlabel("年龄段")
ax[1].bar(eda_price.price_band.astype(str), eda_price["浏览下单转化率_%"], color="#7209B7")
for i, v in enumerate(eda_price["浏览下单转化率_%"]):
    ax[1].text(i, v + 0.4, f"{v}%", ha="center", fontsize=9)
ax[1].set_title("不同商品价格带 浏览→下单 转化率"); ax[1].set_ylabel("转化率（%）"); ax[1].set_xlabel("商品标价区间")
plt.setp(ax[1].get_xticklabels(), rotation=15)
fig.suptitle("EDA：人口属性与商品价格维度的转化差异", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG / "fig08_EDA年龄与价格带转化.png", dpi=150, bbox_inches="tight"); plt.show()
# 说明：图 9~11（Power BI 三页看板）为在 Power BI Desktop 中手工搭建的实物看板，
#      截图与源文件存放于根目录 bi_screenshots/（bi_01~bi_03、.pbix），搭建步骤见 report/PowerBI搭建指南.md。
print("8 张分析图（fig01~fig08）已输出至", FIG)

8 张分析图（fig01~fig08）已输出至 D:\ecom_analysis\figures


## 模块 24｜PowerBI 聚合结果集导出（`output/bi/`，UTF-8 BOM）
本格目标：把看板所需事实表/维度表输出为可直接导入的扁平聚合 CSV。

In [28]:
# 整体漏斗补「全网」行后与分平台漏斗合并，便于单表筛选
_fo = S["funnel_overall"].copy()
_fo.insert(0, "platform_code", "ALL"); _fo.insert(1, "platform_name", "全网")
funnel_all = pd.concat([_fo, S["funnel_platform"]], ignore_index=True)

bi_exports = {
    "fct_platform_metrics.csv": S["core_metrics"],
    "fct_funnel.csv": funnel_all,
    "fct_monthly_gmv.csv": S["monthly_gmv"],
    "fct_rfm_summary.csv": S["rfm_summary"],
    "dim_user_rfm.csv": S["rfm_detail"],
    "fct_category_sales.csv": S["category_sales"],
    "fct_category_funnel.csv": S["category_funnel"],
    "dim_sku_label.csv": S["product_label"],
    "fct_sku_label_summary.csv": S["product_label_summary"],
    "fct_cohort_retention.csv": S["cohort_retention"],
    "fct_platform_level.csv": S["platform_level_compare"],
    "fct_level_metrics.csv": S["level_metrics"],
}
for fname, dfx in bi_exports.items():
    dfx.to_csv(BIDIR / fname, index=False, encoding="utf-8-sig")
print("已导出", len(bi_exports), "张 PowerBI 聚合表：")
for fname in bi_exports:
    print(" -", fname)

已导出 12 张 PowerBI 聚合表：
 - fct_platform_metrics.csv
 - fct_funnel.csv
 - fct_monthly_gmv.csv
 - fct_rfm_summary.csv
 - dim_user_rfm.csv
 - fct_category_sales.csv
 - fct_category_funnel.csv
 - dim_sku_label.csv
 - fct_sku_label_summary.csv
 - fct_cohort_retention.csv
 - fct_platform_level.csv
 - fct_level_metrics.csv


## 模块 24B｜扩展专题一：未完成订单下钻（全量订单口径）

**口径与边界**
- 范围：全部 **10,996** 个订单（区别于 GMV 仅取"已完成"的口径）；金额取 `total_amount`。
- 金额勾稽：`total_amount = Σ order_item.item_total`（全单最大误差 < 1e-12）；`discount_amount / shipping_fee / tax_amount` 不参与该勾稽（独立生成字段，见模块 24C）。
- 状态三分组：**流失类**=待付款+已取消；**在途类**=已付款+已发货；已完成。
- 数据特性提示：`payment_time` 在待付款/已取消订单中也 100% 有值，各状态订单量近似均匀（各约 20%），故支付时效字段不能作为真实流失预警依据，仅做描述。
- 产物：`order_status_overview.csv`、`order_status_platform.csv`、`order_status_amount_band.csv`、`order_status_category.csv`、BI `fct_order_status.csv`、`fig12`。

In [29]:
PLAT_MAP = {"taobao": "淘宝", "jd": "京东", "douyin": "抖音"}
# ============================================================
# 扩展专题一：未完成订单下钻（全量 10,996 单）
# ============================================================
from scipy.stats import chi2_contingency

ex_od = pd.read_csv(DATA / "order.csv", parse_dates=["order_time", "payment_time"])
ex_oi = pd.read_csv(DATA / "order_item.csv")
ex_pr = pd.read_csv(DATA / "product.csv")
ex_N = len(ex_od)
ex_od["pay_lag_min"] = (ex_od.payment_time - ex_od.order_time).dt.total_seconds() / 60
ex_od["amt_band"] = pd.cut(ex_od.total_amount, [0, 1000, 2000, 3000, 4000, 5001],
                           labels=["<1000", "1000-2000", "2000-3000", "3000-4000", "4000-5000"])

T1_STATUS = ["待付款", "已取消", "已付款", "已发货", "已完成"]
T1_CODE = {"待付款": "pending", "已取消": "cancelled", "已付款": "paid",
           "已发货": "shipped", "已完成": "completed"}
T1_GROUP = {"待付款": "流失类", "已取消": "流失类", "已付款": "在途类",
            "已发货": "在途类", "已完成": "已完成"}

# 1) 状态总览
t1_ov = ex_od.groupby("order_status").agg(
    order_cnt=("order_id", "count"), user_cnt=("global_user_id", "nunique"),
    amount=("total_amount", "sum"), avg_amount=("total_amount", "mean"),
    avg_discount=("discount_amount", "mean"), avg_pay_lag_min=("pay_lag_min", "mean")
).reindex(T1_STATUS).reset_index()
t1_ov["order_pct"] = (t1_ov.order_cnt / ex_N * 100).round(2)
t1_ov["amount_wan"] = (t1_ov.amount / 10000).round(1)
t1_ov["amount_pct"] = (t1_ov.amount / t1_ov.amount.sum() * 100).round(2)
t1_ov["status_code"] = t1_ov.order_status.map(T1_CODE)
t1_ov["status_group"] = t1_ov.order_status.map(T1_GROUP)
t1_ov_out = t1_ov[["status_code", "order_status", "status_group", "order_cnt", "order_pct",
                   "user_cnt", "amount_wan", "amount_pct", "avg_amount", "avg_discount",
                   "avg_pay_lag_min"]].round(2)
t1_ov_out.to_csv(MET / "order_status_overview.csv", index=False, encoding="utf-8-sig")
display(t1_ov_out)

# 2) 平台 × 状态
t1_ps = pd.crosstab(ex_od.platform, ex_od.order_status).reindex(columns=T1_STATUS)
t1_ps_out = t1_ps.reset_index().melt(id_vars="platform", var_name="order_status",
                                     value_name="order_cnt")
t1_ps_out["platform_name"] = t1_ps_out.platform.map(PLAT_MAP)
t1_ps_out["platform_order_cnt"] = t1_ps_out.platform.map(t1_ps.sum(axis=1))
t1_ps_out["pct_in_platform"] = (t1_ps_out.order_cnt / t1_ps_out.platform_order_cnt * 100).round(2)
t1_ps_out[["platform", "platform_name", "order_status", "order_cnt", "pct_in_platform"]] \
    .to_csv(MET / "order_status_platform.csv", index=False, encoding="utf-8-sig")

# 3) 金额带 × 状态
t1_ab = pd.crosstab(ex_od.amt_band, ex_od.order_status).reindex(columns=T1_STATUS)
t1_ab_out = t1_ab.reset_index().melt(id_vars="amt_band", var_name="order_status",
                                     value_name="order_cnt")
t1_ab_out["band_order_cnt"] = t1_ab_out.amt_band.astype(str).map(dict(t1_ab.sum(axis=1))).astype(float)
t1_ab_out["pct_in_band"] = (t1_ab_out.order_cnt / t1_ab_out.band_order_cnt * 100).round(2)
t1_ab_out.rename(columns={"amt_band": "amount_band"}).to_csv(
    MET / "order_status_amount_band.csv", index=False, encoding="utf-8-sig")

# 4) 品类 × 状态（明细金额，万元）
t1_oi_j = ex_oi.merge(ex_pr[["global_product_id", "category"]], on="global_product_id", how="left")
t1_cat = (t1_oi_j.merge(ex_od[["order_id", "order_status"]], on="order_id")
          .pivot_table(index="category", columns="order_status", values="item_total",
                       aggfunc="sum", fill_value=0).reindex(columns=T1_STATUS).reset_index()
          .melt(id_vars="category", var_name="order_status", value_name="item_amount"))
t1_cat["amount_wan"] = (t1_cat.item_amount / 10000).round(2)
t1_cat[["category", "order_status", "amount_wan"]].to_csv(
    MET / "order_status_category.csv", index=False, encoding="utf-8-sig")

# 5) BI 聚合表（含全网 ALL 行）
t1_bi = t1_ps_out[["platform", "platform_name", "order_status", "order_cnt", "pct_in_platform"]].copy()
t1_bi.columns = ["platform_code", "platform_name", "order_status", "order_cnt", "status_pct"]
t1_all = t1_ov[["order_status", "order_cnt", "order_pct"]].copy()
t1_all.columns = ["order_status", "order_cnt", "status_pct"]
t1_all.insert(0, "platform_code", "ALL"); t1_all.insert(1, "platform_name", "全网")
t1_bi = pd.concat([t1_all, t1_bi], ignore_index=True)
t1_bi["status_code"] = t1_bi.order_status.map(T1_CODE)
t1_bi.to_csv(BIDIR / "fct_order_status.csv", index=False, encoding="utf-8-sig")

# 6) fig12
_t1_cmap = {"已完成": "#2E8B57", "待付款": "#E18A00", "已取消": "#C0392B",
            "已付款": "#5B8FF9", "已发货": "#7E57C2"}
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
bars = ax[0].bar(t1_ov.order_status, t1_ov.order_cnt,
                 color=[_t1_cmap[s] for s in t1_ov.order_status])
for b, v, pct in zip(bars, t1_ov.order_cnt, t1_ov.order_pct):
    ax[0].text(b.get_x() + b.get_width() / 2, v + 30, f"{v:,}\n{pct}%", ha="center", fontsize=9)
ax[0].set_title("全部订单状态分布（n=10,996）"); ax[0].set_ylabel("订单数（单）"); ax[0].set_ylim(0, 2600)
_t1_sp = t1_ps.div(t1_ps.sum(axis=1), axis=0) * 100
bottom = np.zeros(3)
for st in T1_STATUS:
    ax[1].bar(_t1_sp.index.map(PLAT_MAP), _t1_sp[st], bottom=bottom, label=st,
              color=_t1_cmap[st])
    for i, v in enumerate(_t1_sp[st]):
        ax[1].text(i, bottom[i] + v / 2, f"{v:.1f}%", ha="center", va="center",
                   fontsize=8, color="white")
    bottom += _t1_sp[st].values
ax[1].set_title("各平台订单状态构成（100% 堆叠）"); ax[1].set_ylabel("占比 %"); ax[1].set_ylim(0, 100)
ax[1].legend(ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.08))
plt.tight_layout(); plt.savefig(FIG / "fig12_订单状态分布与平台构成.png", dpi=150, bbox_inches="tight"); plt.show()

print("流失类(待付款+已取消)：", int(t1_ov.loc[t1_ov.status_group=="流失类","order_cnt"].sum()),
      "单 /", t1_ov.loc[t1_ov.status_group=="流失类","amount_wan"].sum(), "万元；",
      "在途类：", int(t1_ov.loc[t1_ov.status_group=="在途类","order_cnt"].sum()),
      "单 /", t1_ov.loc[t1_ov.status_group=="在途类","amount_wan"].sum(), "万元")


,status_code,order_status,status_group,order_cnt,order_pct,user_cnt,amount_wan,amount_pct,avg_amount,avg_discount,avg_pay_lag_min
0,pending,待付款,流失类,2224,20.2300,1634,566.9000,20.2800,"2,549.1900",380.9100,32.4100
1,cancelled,已取消,流失类,2177,19.8000,1648,547.2000,19.5700,"2,513.6200",372.2600,32.1400
2,paid,已付款,在途类,2192,19.9300,1607,563.0000,20.1400,"2,568.5600",385.1800,32.6200
3,shipped,已发货,在途类,2187,19.8900,1631,549.7000,19.6600,"2,513.4200",369.6300,31.8400
4,completed,已完成,已完成,2216,20.1500,1643,569.2000,20.3600,"2,568.6300",383.6000,33.0600


流失类(待付款+已取消)： 4401 单 / 1114.1 万元； 在途类： 4379 单 / 1112.7 万元


## 模块 24C｜扩展专题二：促销 / 优惠券 / 折扣专题

**字段实况（先验收再分析）**
- `promotion_id` 100% 填充（含"无促销"2,212 单），取值 5 类各约 20%；`coupon_code` 5,476 单非空（49.8%）；`discount_amount` 全部 > 0（最小 0.01 元，含"无促销"单）。
- 自定义折扣深度 = `discount_amount / (total_amount + discount_amount)`；由于 discount 与 total 不勾稽，该指标仅用于组间相对比较，不作真实成交价折扣解读。
- 成交/标价比率 = `order_item.unit_price / product.price`（明细行口径）。
- 卡方检验与 Spearman 相关仅用于判断"状态与促销字段是否相互独立"，**属观察性检验，不是 A/B 实验**（无随机分组），不得解读为促销因果增量。
- 产物：`promo_type_summary.csv`、`coupon_summary.csv`、`discount_realized_price.csv`、`topic_stat_tests.csv`（含专题一的独立性检验）、BI `fct_promo_summary.csv`、`fig13`。

In [30]:
# ============================================================
# 扩展专题二：促销 / 优惠券 / 折扣
# ============================================================
from scipy.stats import spearmanr

ex_od["gross"] = ex_od.total_amount + ex_od.discount_amount
ex_od["depth_pct"] = ex_od.discount_amount / ex_od.gross * 100
ex_od["has_coupon"] = np.where(ex_od.coupon_code.notna(), "用券", "无券")

def _promo_block(df):
    g = df.groupby("promotion_id").agg(
        order_cnt=("order_id", "count"), avg_amount=("total_amount", "mean"),
        avg_discount=("discount_amount", "mean"), avg_depth_pct=("depth_pct", "mean"),
        completed_cnt=("order_status", lambda s: (s == "已完成").sum()))
    g["completed_gmv"] = [df.loc[(df.promotion_id == k) & (df.order_status == "已完成"),
                                 "total_amount"].sum() for k in g.index]
    g["order_pct"] = (g.order_cnt / len(df) * 100).round(2)
    g["completion_pct"] = (g.completed_cnt / g.order_cnt * 100).round(2)
    return g.round(2).reset_index()

t2_promo = _promo_block(ex_od)
t2_promo.to_csv(MET / "promo_type_summary.csv", index=False, encoding="utf-8-sig")
display(t2_promo)

# 用券 vs 无券（全网 + 分平台）
t2_rows = []
for scope, sub in [("全网", ex_od)] + [(PLAT_MAP[p], ex_od[ex_od.platform == p])
                                      for p in ["taobao", "jd", "douyin"]]:
    for gp in ["用券", "无券"]:
        sdf = sub[sub.has_coupon == gp]
        t2_rows.append([scope, gp, len(sdf), round(sdf.total_amount.mean(), 2),
                        round(sdf.depth_pct.mean(), 2),
                        round((sdf.order_status == "已完成").mean() * 100, 2)])
t2_coupon = pd.DataFrame(t2_rows, columns=["scope", "coupon_flag", "order_cnt",
                                           "avg_amount", "avg_depth_pct", "completion_pct"])
t2_coupon.to_csv(MET / "coupon_summary.csv", index=False, encoding="utf-8-sig")
display(t2_coupon)

# 成交价 / 标价比率（数据质量发现）
t1_oi_j = t1_oi_j.merge(ex_pr[["global_product_id", "price"]], on="global_product_id", how="left") \
    if "price" not in t1_oi_j.columns else t1_oi_j
t1_oi_j["ratio"] = t1_oi_j.unit_price / t1_oi_j.price
_rp = t1_oi_j.groupby("category").ratio.agg(
    lines="count", p05=lambda s: s.quantile(.05), median="median", p95=lambda s: s.quantile(.95))
_rp["below_list_pct"] = t1_oi_j.assign(b=(t1_oi_j.ratio < 0.999)).groupby("category").b.mean() * 100
_rp["above_list_pct"] = t1_oi_j.assign(b=(t1_oi_j.ratio > 1.001)).groupby("category").b.mean() * 100
t2_rp = pd.concat([pd.DataFrame({"category": ["全网"], "lines": [len(t1_oi_j)],
                                 "p05": [t1_oi_j.ratio.quantile(.05)], "median": [t1_oi_j.ratio.median()],
                                 "p95": [t1_oi_j.ratio.quantile(.95)],
                                 "below_list_pct": [(t1_oi_j.ratio < 0.999).mean() * 100],
                                 "above_list_pct": [(t1_oi_j.ratio > 1.001).mean() * 100]}),
                   _rp.reset_index()], ignore_index=True).round(3)
t2_rp.to_csv(MET / "discount_realized_price.csv", index=False, encoding="utf-8-sig")
display(t2_rp)

# BI：促销汇总（全网 + 平台）
t2_bi = t2_promo.copy(); t2_bi.insert(0, "platform_code", "ALL"); t2_bi.insert(1, "platform_name", "全网")
for p in ["taobao", "jd", "douyin"]:
    g = _promo_block(ex_od[ex_od.platform == p])
    g.insert(0, "platform_code", p); g.insert(1, "platform_name", PLAT_MAP[p])
    t2_bi = pd.concat([t2_bi, g], ignore_index=True)
t2_bi.to_csv(BIDIR / "fct_promo_summary.csv", index=False, encoding="utf-8-sig")

# 独立性检验（专题一 + 专题二合并一张证据表）
t2_tests = []
for col, name in [("platform", "订单状态×平台"), ("promotion_id", "订单状态×促销类型"),
                  ("amt_band", "订单状态×金额带"), ("has_coupon", "订单状态×是否用券")]:
    ct = pd.crosstab(ex_od.order_status, ex_od[col])
    chi2, pv, dof, _ = chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    t2_tests.append([name, "chi2", round(chi2, 2), dof, round(pv, 4), round(v, 4),
                     "独立（p>0.05）" if pv > 0.05 else "有关联"])
ex_od["done"] = (ex_od.order_status == "已完成").astype(int)
rho1, p1 = spearmanr(ex_od.depth_pct, ex_od.done)
rho2, p2 = spearmanr(ex_od.depth_pct, ex_od.total_amount)
t2_tests.append(["折扣深度×是否完成", "spearman", round(rho1, 4), np.nan, round(p1, 4), np.nan,
                 "无相关" if p1 > 0.05 else "弱相关"])
t2_tests.append(["折扣深度×订单金额", "spearman", round(rho2, 4), np.nan, round(p2, 4), np.nan,
                 "效应量极小（rho≈-0.02），无业务意义"])
t2_stat = pd.DataFrame(t2_tests, columns=["test_pair", "test", "statistic", "dof",
                                          "p_value", "cramers_v_or_rho", "conclusion"])
t2_stat.to_csv(MET / "topic_stat_tests.csv", index=False, encoding="utf-8-sig")
display(t2_stat)
_t2_p_promo = float(t2_stat.loc[t2_stat.test_pair == "订单状态×促销类型", "p_value"].iloc[0])

# fig13
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
_po = t2_promo.set_index("promotion_id").reindex(["无促销", "满减活动", "优惠券", "拼团优惠", "秒杀活动"])
x = np.arange(len(_po))
ax[0].bar(x, _po.completion_pct, color=["#7F8C8D", "#5B8FF9", "#E18A00", "#2E8B57", "#C0392B"])
ax[0].set_xticks(x); ax[0].set_xticklabels(_po.index)
ax[0].set_ylim(0, 26); ax[0].set_ylabel("已完成订单占比 %")
ax[0].set_title(f"各促销类型订单完成率（卡方 p={_t2_p_promo:.3f}，无显著差异）")
for i, v in enumerate(_po.completion_pct):
    ax[0].text(i, v + 0.4, f"{v}%", ha="center", fontsize=9)
ax0b = ax[0].twinx()
ax0b.plot(x, _po.avg_depth_pct, "o--", color="#34495E", linewidth=1.5)
ax0b.set_ylabel("平均折扣深度 %"); ax0b.set_ylim(0, 20)
_rclip = t1_oi_j.ratio.clip(0, 2)
ax[1].hist(_rclip, bins=60, color="#5B8FF9", edgecolor="white")
ax[1].axvline(1, color="#C0392B", linestyle="--", linewidth=1.5)
ax[1].text(1.02, ax[1].get_ylim()[1] * 0.85, "标价=1", color="#C0392B", fontsize=9)
ax[1].set_title("成交价/标价比率分布（裁剪至[0,2]；实际范围0.002~70.97，中位0.38）")
ax[1].set_xlabel("unit_price / list_price"); ax[1].set_ylabel("明细行数")
plt.tight_layout(); plt.savefig(FIG / "fig13_促销完成率与成交标价比率.png", dpi=150, bbox_inches="tight"); plt.show()


,promotion_id,order_cnt,avg_amount,avg_discount,avg_depth_pct,completed_cnt,completed_gmv,order_pct,completion_pct
0,优惠券,2147,"2,530.5600",381.5600,12.5600,442,"1,127,160.2300",19.5300,20.5900
1,拼团优惠,2232,"2,581.1100",379.9600,12.4100,425,"1,101,473.1800",20.3000,19.0400
2,无促销,2212,"2,540.3400",382.0600,12.7100,466,"1,186,801.8400",20.1200,21.0700
3,满减活动,2203,"2,511.9600",376.1700,12.6400,425,"1,074,451.5400",20.0300,19.2900
4,秒杀活动,2202,"2,549.2800",372.0200,12.4100,458,"1,202,205.3400",20.0300,20.8000


,scope,coupon_flag,order_cnt,avg_amount,avg_depth_pct,completion_pct
0,全网,用券,5476,"2,549.2100",12.4500,20.1400
1,全网,无券,5520,"2,536.4700",12.6400,20.1600
2,淘宝,用券,2246,"2,570.9800",12.0800,20.1200
3,淘宝,无券,2254,"2,573.7300",12.6900,20.4100
4,京东,用券,1589,"2,532.1500",12.6800,19.9500
5,京东,无券,1606,"2,470.7300",12.8200,19.5500
6,抖音,用券,1641,"2,535.9200",12.7300,20.3500
7,抖音,无券,1660,"2,549.4800",12.4100,20.4200


,category,lines,p05,median,p95,below_list_pct,above_list_pct
0,全网,27523,0.0430,0.3780,3.6470,79.7040,20.2590
1,图书文具,3510,0.0410,0.3450,2.4390,82.4790,17.4930
2,家居用品,3373,0.0470,0.3890,3.7600,78.0610,21.9090
3,服装鞋帽,3348,0.0440,0.3710,3.2600,80.7650,19.1760
4,母婴用品,3462,0.0390,0.3860,3.4330,80.1560,19.7280
5,电子产品,3341,0.0440,0.3710,4.7070,79.5570,20.4130
6,美妆个护,3373,0.0410,0.3790,3.9890,79.2170,20.7830
7,运动户外,3314,0.0420,0.3750,3.6920,80.8690,19.1310
8,食品饮料,3802,0.0490,0.4100,4.7240,76.8020,23.1720


,test_pair,test,statistic,dof,p_value,cramers_v_or_rho,conclusion
0,订单状态×平台,chi2,4.5200,8.0000,0.8073,0.0143,独立（p>0.05）
1,订单状态×促销类型,chi2,21.9500,16.0000,0.1449,0.0223,独立（p>0.05）
2,订单状态×金额带,chi2,17.5300,16.0000,0.3519,0.0200,独立（p>0.05）
3,订单状态×是否用券,chi2,0.1600,4.0000,0.9968,0.0039,独立（p>0.05）
4,折扣深度×是否完成,spearman,0.0057,NaN,0.5525,NaN,无相关
5,折扣深度×订单金额,spearman,-0.0224,NaN,0.0186,NaN,效应量极小（rho≈-0.02），无业务意义


## 模块 24D｜扩展专题三：购物篮连带分析

**结构事实与口径**
- 全部 27,523 个明细行的 `quantity` 恒等于 1；每单 2~3 行（2 行 5,465 单、3 行 5,531 单）→ **件数 = 明细行数**。
- UPT（件/单）= 总件数 / 订单数；件单价 = 金额 / 总件数；跨品类订单占比 = 含 ≥2 个一级品类的订单占比。
- 同时给两种口径：**全量订单**与**已完成订单**（与 GMV 口径对齐，可与客单价 2,568.63 元互相勾稽：UPT×件单价≈AOV）。
- 品类对关联规则：support=P(A∩B)、confidence=P(B|A)、lift=P(B|A)/P(B)。因每单仅抽 2~3 个品类，lift 机械上 <1，是否存在连带结构看 28 对 lift 的**相对离散度**而非绝对值。
- 产物：`basket_metrics.csv`、`category_pair_lift.csv`、BI `fct_category_pair.csv`、`fig14`。

In [31]:
# ============================================================
# 扩展专题三：购物篮连带
# ============================================================
from itertools import combinations

# 双口径篮子指标：全网 + 分平台
t3_rows = []
for scope, sub_ord in [("全网", ex_od)] + [(PLAT_MAP[p], ex_od[ex_od.platform == p])
                                          for p in ["taobao", "jd", "douyin"]]:
    for cal, ids in [("全量订单", set(sub_ord.order_id)),
                     ("已完成订单", set(sub_ord.loc[sub_ord.order_status == "已完成", "order_id"]))]:
        d = t1_oi_j[t1_oi_j.order_id.isin(ids)]
        oc = d.groupby("order_id").agg(lines=("order_item_id", "count"), cats=("category", "nunique"))
        n = len(oc)
        t3_rows.append([scope, cal, n, int(d.quantity.sum()), round(oc.lines.mean(), 3),
                        round(d.quantity.sum() / n, 3),
                        round(d.item_total.sum() / d.quantity.sum(), 2),
                        round((oc.cats >= 2).mean() * 100, 2)])
t3_basket = pd.DataFrame(t3_rows, columns=["scope", "caliber", "order_cnt", "units",
                                           "lines_per_order", "upt_units_per_order",
                                           "price_per_unit", "cross_category_order_pct"])
t3_basket.to_csv(MET / "basket_metrics.csv", index=False, encoding="utf-8-sig")
display(t3_basket)

# 品类对 support / confidence / lift（全量订单，8 品类 28 对）
t3_order_cats = t1_oi_j.groupby("order_id").category.apply(lambda s: set(s))
t3_cats = sorted(t1_oi_j.category.dropna().unique())
t3_prows = []
for a, b in combinations(t3_cats, 2):
    sa = t3_order_cats.apply(lambda s: a in s)
    sb = t3_order_cats.apply(lambda s: b in s)
    sab = int((sa & sb).sum())
    t3_prows.append([a, b, sab, round(sab / ex_N * 100, 2),
                     round(sab / sa.sum() * 100, 2), round(sab / sb.sum() * 100, 2),
                     round((sab / sa.sum()) / (sb.sum() / ex_N), 3)])
t3_pairs = pd.DataFrame(t3_prows, columns=["category_a", "category_b", "pair_orders",
                                           "support_pct", "confidence_ab_pct",
                                           "confidence_ba_pct", "lift"])
t3_pairs.to_csv(MET / "category_pair_lift.csv", index=False, encoding="utf-8-sig")
t3_pairs.to_csv(BIDIR / "fct_category_pair.csv", index=False, encoding="utf-8-sig")
print("品类对 lift：min", t3_pairs.lift.min(), "max", t3_pairs.lift.max(),
      "mean", round(t3_pairs.lift.mean(), 3), "std", round(t3_pairs.lift.std(), 3))
display(t3_pairs.sort_values("lift", ascending=False).head(5))

# fig14：lift 热力图 + UPT/件单价
fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.4), gridspec_kw={"width_ratios": [1.15, 1]})
t3_mat = pd.DataFrame(np.ones((len(t3_cats), len(t3_cats))), index=t3_cats, columns=t3_cats)
for _, r in t3_pairs.iterrows():
    t3_mat.loc[r.category_a, r.category_b] = r.lift
    t3_mat.loc[r.category_b, r.category_a] = r.lift
sns.heatmap(t3_mat, annot=True, fmt=".2f", cmap="Blues", ax=ax[0], vmin=0.6, vmax=1.0,
            linewidths=.5, cbar_kws={"label": "lift"})
ax[0].set_title("品类共购 lift 热力图（全部<1 且集中于0.67~0.76，无连带结构）")
_bd = t3_basket[t3_basket.caliber == "已完成订单"].set_index("scope").reindex(["全网", "淘宝", "京东", "抖音"])
x2 = np.arange(len(_bd))
ax[1].bar(x2 - 0.2, _bd.upt_units_per_order, width=0.4, label="UPT（件/单）", color="#5B8FF9")
ax[1].set_xticks(x2); ax[1].set_xticklabels(_bd.index); ax[1].set_ylim(0, 3.2); ax[1].set_ylabel("UPT（件/单）")
for i, v in enumerate(_bd.upt_units_per_order):
    ax[1].text(i - 0.2, v + 0.03, f"{v:.2f}", ha="center", fontsize=9)
ax1b = ax[1].twinx()
ax1b.bar(x2 + 0.2, _bd.price_per_unit, width=0.4, label="件单价（元）", color="#FF9F40")
ax1b.set_ylabel("件单价（元）"); ax1b.set_ylim(0, 2500)
for i, v in enumerate(_bd.price_per_unit):
    ax1b.text(i + 0.2, v + 30, f"{v:.0f}", ha="center", fontsize=9)
h1, l1 = ax[1].get_legend_handles_labels(); h2, l2 = ax1b.get_legend_handles_labels()
ax[1].legend(h1 + h2, l1 + l2, loc="upper right", fontsize=9)
ax[1].set_title("已完成订单 UPT 与件单价（数量恒为1，件数=明细行数）")
plt.tight_layout(); plt.savefig(FIG / "fig14_品类共购lift与UPT.png", dpi=150, bbox_inches="tight"); plt.show()


,scope,caliber,order_cnt,units,lines_per_order,upt_units_per_order,price_per_unit,cross_category_order_pct
0,全网,全量订单,10996,27523,2.5030,2.5030,"1,015.9100",93.3700
1,全网,已完成订单,2216,5576,2.5160,2.5160,"1,020.8200",93.7700
2,淘宝,全量订单,4500,11264,2.5030,2.5030,"1,027.6600",93.1800
3,淘宝,已完成订单,912,2306,2.5290,2.5290,"1,007.3700",93.6400
4,京东,全量订单,3195,7986,2.5000,2.5000,"1,000.7000",92.9300
5,京东,已完成订单,631,1590,2.5200,2.5200,"1,016.6200",93.9800
6,抖音,全量订单,3301,8273,2.5060,2.5060,"1,014.5800",94.0600
7,抖音,已完成订单,673,1680,2.4960,2.4960,"1,043.2600",93.7600


品类对 lift：min 0.674 max 0.763 mean 0.712 std 0.023


,category_a,category_b,pair_orders,support_pct,confidence_ab_pct,confidence_ba_pct,lift
11,家居用品,运动户外,641,5.8300,20.8700,21.3100,0.7630
21,母婴用品,食品饮料,739,6.7200,23.4800,21.6400,0.7560
0,图书文具,家居用品,660,6.0000,21.0100,21.4800,0.7520
18,母婴用品,电子产品,641,5.8300,20.3700,21.1100,0.7380
15,服装鞋帽,美妆个护,621,5.6500,20.3300,20.3900,0.7340


## 模块 25｜阶段2 补充：口径衔接（行为表「下单」× 订单表下单）

In [32]:
# ============================================================
# 模块 25｜口径衔接：行为表「下单」用户 × 订单表下单用户
# 漏斗取自 user_behavior（行为日志），GMV/购买用户取自 order（交易表），
# 两套口径的「下单」不是同一件事，本表把差异量化后公开，避免跨节误引用。
# ============================================================
def _overlap(scope_code):
    """scope_code=None 表示全网；返回两个口径的用户集合关系"""
    if scope_code is None:
        a = set(user_behavior.loc[user_behavior.behavior_type == "下单", "global_user_id"])
        bo = set(order.global_user_id)
        bc = set(completed.global_user_id)
    else:
        _b = user_behavior[(user_behavior.behavior_type == "下单") & (user_behavior.platform == scope_code)]
        a = set(_b.global_user_id)
        bo = set(order.loc[order.platform == scope_code, "global_user_id"])
        bc = set(completed.loc[completed.platform == scope_code, "global_user_id"])
    return {
        "behavior_order_users": len(a),        # 行为表出现「下单」事件的去重用户
        "order_users_any": len(bo),            # 订单表任意状态下单的去重用户
        "order_users_completed": len(bc),      # 订单表已完成订单的去重用户
        "both_behavior_and_any": len(a & bo),
        "both_behavior_and_completed": len(a & bc),
        "behavior_only": len(a - bo),          # 有下单行为、订单表查无任何单
        "order_only": len(bo - a),             # 订单表有单、行为表无下单事件
        "behavior_match_any_pct": round(100 * len(a & bo) / len(a), 2),
        "behavior_match_completed_pct": round(100 * len(a & bc) / len(a), 2),
    }

_P3 = {"taobao": "淘宝", "jd": "京东", "douyin": "抖音"}
_rows = [dict(scope_code="ALL", scope="全网", **_overlap(None))]
for _pc, _pn in _P3.items():
    _rows.append(dict(scope_code=_pc, scope=_pn, **_overlap(_pc)))
behavior_order_overlap = pd.DataFrame(_rows)[
    ["scope_code", "scope", "behavior_order_users", "order_users_any", "order_users_completed",
     "both_behavior_and_any", "both_behavior_and_completed", "behavior_only", "order_only",
     "behavior_match_any_pct", "behavior_match_completed_pct"]]
behavior_order_overlap.to_csv(MET / "behavior_order_overlap.csv", index=False, encoding="utf-8-sig")
behavior_order_overlap


,scope_code,scope,behavior_order_users,order_users_any,order_users_completed,both_behavior_and_any,both_behavior_and_completed,behavior_only,order_only,behavior_match_any_pct,behavior_match_completed_pct
0,ALL,全网,1953,4215,1643,1551,640,402,2664,79.4200,32.7700
1,taobao,淘宝,622,1301,630,562,284,60,739,90.3500,45.6600
2,jd,京东,661,1217,444,523,193,138,694,79.1200,29.2000
3,douyin,抖音,670,1697,569,466,163,204,1231,69.5500,24.3300


## 模块 26｜阶段3 补充：逻辑回归基线对比与意图先验强度

In [33]:
# ============================================================
# 模块 26｜逻辑回归基线对比 + 意图信号先验强度
# 目的：把 3.6 节的单模型扩展为「基线对照」，说清 AUC 0.78 的判别力来自哪几类特征；
#       并给出「有加购/有收藏」的先验下单率，避免把前置行为与标签的同向关系写成发现。
# 说明：全部为训练集（全样本）样本内评估，无交叉验证，不作为泛化或预测依据。
# ============================================================
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.metrics import roc_auc_score as _auc

_pv = user_behavior.pivot_table(index="global_user_id", columns="behavior_type",
                                values="behavior_id", aggfunc="count", fill_value=0)
_bs = set(user_behavior.loc[user_behavior.behavior_type == "浏览", "global_user_id"])
_os = set(user_behavior.loc[user_behavior.behavior_type == "下单", "global_user_id"])
_lrd = (user[user.global_user_id.isin(_bs)]
        .merge(_pv.reset_index(), on="global_user_id", how="left")
        .merge(user_behavior.groupby("global_user_id").duration_seconds.mean().rename("avg_duration"),
               on="global_user_id", how="left"))
# 只填数值型特征列：user 表在前面已被追加过分类型列（如 age_band），整表 fillna 会报错
for _c in list(_pv.columns) + ["avg_duration"]:
    if _c in _lrd.columns:
        _lrd[_c] = _lrd[_c].fillna(0)
_lrd["gender_m"] = (_lrd.gender == "m").astype(int)
_lrd["is_jd"] = (_lrd.platform == "jd").astype(int)
_lrd["is_douyin"] = (_lrd.platform == "douyin").astype(int)
_lrd["level_ord"] = _lrd.user_level.map({"普通会员": 1, "银卡会员": 2, "金卡会员": 3, "钻石会员": 4})
_lrd["y"] = _lrd.global_user_id.isin(_os).astype(int)

_GROUPS = {
    "仅人口属性": ["age", "gender_m", "level_ord"],
    "仅平台": ["is_jd", "is_douyin"],
    "仅意图(收藏+加购)": ["收藏", "加购"],
    "全集(9特征)": ["age", "gender_m", "level_ord", "浏览", "加购", "收藏", "avg_duration", "is_jd", "is_douyin"],
    "全集去意图": ["age", "gender_m", "level_ord", "浏览", "avg_duration", "is_jd", "is_douyin"],
}
_rows = [{"model": "随机基线(常数预测器理论值 0.5)", "features": "无", "feature_cnt": 0,
          "sample_n": len(_lrd), "pos_rate": round(_lrd.y.mean(), 4), "train_auc": 0.5}]
for _name, _cols in _GROUPS.items():
    _X = _SS().fit_transform(_lrd[_cols].astype(float))
    _m = _LR(max_iter=1000).fit(_X, _lrd.y)
    _rows.append({"model": _name, "features": " + ".join(_cols), "feature_cnt": len(_cols),
                  "sample_n": len(_lrd), "pos_rate": round(_lrd.y.mean(), 4),
                  "train_auc": round(_auc(_lrd.y, _m.predict_proba(_X)[:, 1]), 4)})
lr_baseline = pd.DataFrame(_rows)
lr_baseline.to_csv(MET / "lr_baseline.csv", index=False, encoding="utf-8-sig")

# 意图信号先验强度：有/无加购、有/无收藏 两组的下单率（说明意图与标签的先行关系）
_rows = []
for _f in ["加购", "收藏"]:
    for _has in (True, False):
        _sub = _lrd[_lrd[_f] > 0] if _has else _lrd[_lrd[_f] == 0]
        _rows.append({"signal": _f, "group": ("有" if _has else "无") + _f, "users": len(_sub),
                      "order_users": int(_sub.y.sum()),
                      "order_rate_pct": round(100 * _sub.y.mean(), 2)})
lr_intent_prior = pd.DataFrame(_rows)
lr_intent_prior.to_csv(MET / "lr_intent_prior.csv", index=False, encoding="utf-8-sig")
display(lr_baseline)
display(lr_intent_prior)


,model,features,feature_cnt,sample_n,pos_rate,train_auc
0,随机基线(常数预测器理论值 0.5),无,0,2711,0.4725,0.5000
1,仅人口属性,age + gender_m + level_ord,3,2711,0.4725,0.5124
2,仅平台,is_jd + is_douyin,2,2711,0.4725,0.5429
3,仅意图(收藏+加购),收藏 + 加购,2,2711,0.4725,0.7702
4,全集(9特征),age + gender_m + level_ord + 浏览 + 加购 + 收藏 + av...,9,2711,0.4725,0.7766
5,全集去意图,age + gender_m + level_ord + 浏览 + avg_duration...,7,2711,0.4725,0.6492


,signal,group,users,order_users,order_rate_pct
0,加购,有加购,1740,1005,57.7600
1,加购,无加购,971,276,28.4200
2,收藏,有收藏,1496,1010,67.5100
3,收藏,无收藏,1215,271,22.3000


## 模块 27｜阶段3 补充：时段与星期专题（附录 F）

In [34]:
# ============================================================
# 模块 27｜时段与星期专题（报告附录 F）
# 口径：订单类指标取「已完成」有效订单；行为类指标取 user_behavior 全量行为事件
# ============================================================
_P3 = {"taobao": "淘宝", "jd": "京东", "douyin": "抖音"}

# 1) 分时段（全网 + 分平台，已完成订单口径）
_cocomp = completed.copy()
_cocomp["hour"] = _cocomp.order_time.dt.hour
_cocomp["day_type"] = np.where(_cocomp.order_time.dt.dayofweek >= 5, "周末", "工作日")
_rows = []
for _pc, _pn in [("ALL", "全网")] + list(_P3.items()):
    _d = _cocomp if _pc == "ALL" else _cocomp[_cocomp.platform == _pc]
    _g = _d.groupby("hour").agg(valid_orders=("order_id", "nunique"),
                                gmv=("total_amount", "sum"),
                                buyer_users=("global_user_id", "nunique"))
    for _h in range(24):
        if _h in _g.index:
            _r = _g.loc[_h]
            _rows.append({"scope_code": _pc, "scope": _pn, "hour": _h,
                          "valid_orders": int(_r.valid_orders), "gmv": round(_r.gmv, 2),
                          "buyer_users": int(_r.buyer_users),
                          "order_share_pct": round(100 * _r.valid_orders / _g.valid_orders.sum(), 2),
                          "aov": round(_r.gmv / _r.valid_orders, 2)})
        else:
            _rows.append({"scope_code": _pc, "scope": _pn, "hour": _h, "valid_orders": 0,
                          "gmv": 0.0, "buyer_users": 0, "order_share_pct": 0.0, "aov": 0.0})
time_pattern_order = pd.DataFrame(_rows)
time_pattern_order.to_csv(MET / "time_pattern_order.csv", index=False, encoding="utf-8-sig")

# 2) 行为口径分时段（四环节事件数，全量行为）
_bh = user_behavior.assign(hour=user_behavior.behavior_time.dt.hour)
_pv2 = (_bh[_bh.behavior_type.isin(stages)]
        .pivot_table(index="hour", columns="behavior_type", values="behavior_id",
                     aggfunc="count", fill_value=0)
        .reindex(range(24), fill_value=0).reindex(columns=stages, fill_value=0))
_pv2["行为总数"] = _bh.groupby("hour").size().reindex(range(24), fill_value=0)
time_pattern_behavior = _pv2.reset_index()
time_pattern_behavior.to_csv(MET / "time_pattern_behavior.csv", index=False, encoding="utf-8-sig")

# 3) 工作日 vs 周末（附自然日基准：分析窗口 2025-01-01 ~ 2026-12-28 的日历天数）
_ALL_DAYS = pd.date_range("2025-01-01", "2026-12-28", freq="D")
_DAY_BASE = pd.Series(np.where(_ALL_DAYS.dayofweek >= 5, "周末", "工作日")).value_counts().to_dict()
_N_DAYS = int(sum(_DAY_BASE.values()))
_rows = []
for _pc, _pn in [("ALL", "全网")] + list(_P3.items()):
    _d = _cocomp if _pc == "ALL" else _cocomp[_cocomp.platform == _pc]
    for _dt in ["工作日", "周末"]:
        _s = _d[_d.day_type == _dt]
        _rows.append({"scope_code": _pc, "scope": _pn, "day_type": _dt,
                      "valid_orders": int(_s.order_id.nunique()),
                      "gmv": round(_s.total_amount.sum(), 2),
                      "buyer_users": int(_s.global_user_id.nunique()),
                      "aov": round(_s.total_amount.sum() / max(_s.order_id.nunique(), 1), 2),
                      "order_share_pct": round(100 * _s.order_id.nunique() / max(_d.order_id.nunique(), 1), 2),
                      # 自然日基准：判断「结构占比」是否只反映了工作日/周末的天数比例
                      "calendar_days": _DAY_BASE.get(_dt, 0),
                      "calendar_day_pct": round(100 * _DAY_BASE.get(_dt, 0) / _N_DAYS, 2),
                      "orders_per_calendar_day": round(int(_s.order_id.nunique()) / max(_DAY_BASE.get(_dt, 0), 1), 2)})
time_pattern_weekday = pd.DataFrame(_rows)
time_pattern_weekday.to_csv(MET / "time_pattern_weekday.csv", index=False, encoding="utf-8-sig")
display(time_pattern_weekday)

# fig15：时段专题三面板
_ha = time_pattern_order[time_pattern_order.scope_code == "ALL"].set_index("hour")
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
ax[0].bar(_ha.index, _ha.valid_orders, color="#3A86FF", alpha=.85)
ax[0].set_xlabel("下单小时"); ax[0].set_ylabel("有效订单量（单）"); ax[0].set_xticks(range(0, 24, 3))
ax[0].set_title("全网有效订单的分时段分布")
_ax0b = ax[0].twinx()
_ax0b.plot(_ha.index, _ha.aov, color="#D62828", marker="o", ms=3, lw=1.5)
_ax0b.set_ylabel("客单价（元）"); _ax0b.set_ylim(0, max(_ha.aov.max() * 1.8, 1))
for _pc, _pn in _P3.items():
    _d = time_pattern_order[time_pattern_order.scope_code == _pc].set_index("hour")
    ax[1].plot(_d.index, _d.order_share_pct, marker="o", ms=3, lw=1.6, label=_pn, color=COLORS[_pn])
ax[1].set_xlabel("下单小时"); ax[1].set_ylabel("占该平台有效订单比例（%）")
ax[1].set_xticks(range(0, 24, 3)); ax[1].set_title("分平台下单时段结构占比"); ax[1].legend()
_wk = time_pattern_weekday[time_pattern_weekday.scope_code != "ALL"]
_xw = np.arange(3)
for _i, _dt in enumerate(["工作日", "周末"]):
    _v = [_wk[(_wk.scope_code == k) & (_wk.day_type == _dt)].order_share_pct.iloc[0] for k in _P3]
    _b = ax[2].bar(_xw + (_i - 0.5) * 0.38, _v, width=0.38, label=_dt,
                   color=["#8D99AE", "#E63946"][_i])
    for _bb, _vv in zip(_b, _v):
        ax[2].text(_bb.get_x() + _bb.get_width() / 2, _vv + 0.5, f"{_vv:.1f}%", ha="center", fontsize=8)
ax[2].set_xticks(_xw); ax[2].set_xticklabels(list(_P3.values()))
ax[2].set_ylabel("占该平台有效订单比例（%）"); ax[2].set_ylim(0, 100)
ax[2].set_title("工作日 vs 周末 订单结构占比"); ax[2].legend()
fig.suptitle("时段专题：下单时段分布与工作日/周末结构（已完成订单口径）", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG / "fig15_下单时段与星期结构.png", dpi=150, bbox_inches="tight"); plt.show()


,scope_code,scope,day_type,valid_orders,gmv,buyer_users,aov,order_share_pct,calendar_days,calendar_day_pct,orders_per_calendar_day
0,ALL,全网,工作日,1610,"4,142,692.7800",1272,"2,573.1000",72.6500,519,71.3900,3.1000
1,ALL,全网,周末,606,"1,549,399.3500",552,"2,556.7600",27.3500,208,28.6100,2.9100
2,taobao,淘宝,工作日,653,"1,668,330.6300",499,"2,554.8700",71.6000,519,71.3900,1.2600
3,taobao,淘宝,周末,259,"654,655.0000",231,"2,527.6300",28.4000,208,28.6100,1.2500
4,jd,京东,工作日,470,"1,217,787.6700",336,"2,591.0400",74.4800,519,71.3900,0.9100
5,jd,京东,周末,161,"398,639.1900",143,"2,476.0200",25.5200,208,28.6100,0.7700
6,douyin,抖音,工作日,487,"1,256,574.4800",437,"2,580.2400",72.3600,519,71.3900,0.9400
7,douyin,抖音,周末,186,"496,105.1600",178,"2,667.2300",27.6400,208,28.6100,0.8900


## 模块 28｜阶段3 补充：维度同质性检查（附录 G）

In [35]:
# ============================================================
# 模块 28｜维度同质性检查（报告附录 G）
# 目的：用与附录 C 相同的卡方 + Cramer's V 口径回答「哪些维度有可用信息、哪些是均匀随机」，
#       把「没做品牌/支付方式/配送方式/性别分析」变成「做了筛查并证明该维度无可用信息」。
# 粒度：订单级维度（支付方式/配送方式/性别）取全量订单；商品级维度（品牌/商品分层）取订单明细行；
#       停留时长取行为事件行。Cramer's V<0.05 视为无可用信息，0.05~0.1 为弱关联，>=0.1 为有信息。
# ============================================================
from scipy.stats import chi2_contingency as _chi2

_BANDS = [0, 1000, 2000, 3000, 4000, 5001]
_BLAB = ["<1000", "1000-2000", "2000-3000", "3000-4000", "4000-5000"]
_od2 = order.merge(user[["global_user_id", "gender", "user_level"]], on="global_user_id", how="left")
_od2["amt_band"] = pd.cut(_od2.total_amount, _BANDS, labels=_BLAB)
_od2["done"] = np.where(_od2.order_status == "已完成", "已完成", "未完成")
_oi2 = (order_item.merge(order[["order_id", "order_status", "platform", "total_amount"]], on="order_id")
        .merge(product[["global_product_id", "category", "brand"]], on="global_product_id", how="left")
        .merge(S["product_label"][["global_product_id", "sku_label"]], on="global_product_id", how="left"))
_oi2["amt_band"] = pd.cut(_oi2.total_amount, _BANDS, labels=_BLAB)
_oi2["done"] = np.where(_oi2.order_status == "已完成", "已完成", "未完成")
_bh2 = user_behavior.copy()
_bh2["dur_band"] = pd.qcut(_bh2.duration_seconds.rank(method="first"), 4, labels=["Q1最短", "Q2", "Q3", "Q4最长"])
_bh2f = _bh2[_bh2.behavior_type.isin(stages)]

_hom_rows = []
def _hom(dim_name, dim_vals, out_name, out_vals, grain, n):
    _ct = pd.crosstab(dim_vals, out_vals)
    _c2, _p, _dof, _ = _chi2(_ct)
    _v = float(np.sqrt(_c2 / (_ct.values.sum() * (min(_ct.shape) - 1))))
    _hom_rows.append({"dimension": dim_name, "dim_value_cnt": int(_ct.shape[0]),
                      "test_against": out_name, "grain": grain, "n_obs": int(_ct.values.sum()),
                      "chi2": round(_c2, 2), "dof": int(_dof), "p_value": round(_p, 4),
                      "cramers_v": round(_v, 4),
                      "conclusion": ("有可用信息(V>=0.1)" if _v >= 0.1 else
                                     ("弱关联(0.05<=V<0.1)" if _v >= 0.05 else "无可用信息(V<0.05)"))})

_ORD_DIMS = [("payment_method", _od2.payment_method), ("shipping_method", _od2.shipping_method),
             ("gender", _od2.gender)]
_ITEM_DIMS = [("brand", _oi2.brand), ("sku_label(爆品/动销/滞销)", _oi2.sku_label)]
for _dn, _dv in _ORD_DIMS:
    _hom(_dn, _dv, "订单状态", _od2.order_status, "订单", len(_od2))
    _hom(_dn, _dv, "订单金额带", _od2.amt_band, "订单", len(_od2))
    _hom(_dn, _dv, "平台", _od2.platform, "订单", len(_od2))
    _hom(_dn, _dv, "是否完成", _od2.done, "订单", len(_od2))
for _dn, _dv in _ITEM_DIMS:
    _hom(_dn, _dv, "订单状态", _oi2.order_status, "订单明细行", len(_oi2))
    _hom(_dn, _dv, "订单金额带", _oi2.amt_band, "订单明细行", len(_oi2))
    _hom(_dn, _dv, "平台", _oi2.platform, "订单明细行", len(_oi2))
    _hom(_dn, _dv, "是否完成", _oi2.done, "订单明细行", len(_oi2))
_hom("brand", _oi2.brand, "一级品类", _oi2.category, "订单明细行", len(_oi2))
_hom("sku_label(爆品/动销/滞销)", _oi2.sku_label, "一级品类", _oi2.category, "订单明细行", len(_oi2))
_hom("duration_band", _bh2f.dur_band, "行为环节", _bh2f.behavior_type, "行为事件(四环节)", len(_bh2f))
_hom("duration_band", _bh2.dur_band, "平台", _bh2.platform, "行为事件", len(_bh2))
dim_homogeneity_check = pd.DataFrame(_hom_rows)
# 口径内生标注：sku_label（爆品/动销/滞销）本身由「已完成订单销售额」定义，
# 它与 是否完成 / 订单状态 / 平台 / 一级品类 的检验存在定义循环，不构成独立业务结论
dim_homogeneity_check["note"] = ["口径内生：sku_label 由已完成订单销售额定义，与状态/平台/品类的关联含定义循环，不构成独立业务结论"
                                 if str(_d).startswith("sku_label") else ""
                                 for _d in dim_homogeneity_check.dimension]
dim_homogeneity_check.to_csv(MET / "dim_homogeneity_check.csv", index=False, encoding="utf-8-sig")
display(dim_homogeneity_check)
print("无可用信息(V<0.05)的检验数：", int((dim_homogeneity_check.cramers_v < 0.05).sum()),
      "/", len(dim_homogeneity_check))


,dimension,dim_value_cnt,test_against,grain,n_obs,chi2,dof,p_value,cramers_v,conclusion,note
0,payment_method,5,订单状态,订单,10996,14.5300,16,0.5594,0.0182,无可用信息(V<0.05),
1,payment_method,5,订单金额带,订单,10996,11.7900,16,0.7581,0.0164,无可用信息(V<0.05),
2,payment_method,5,平台,订单,10996,5.6600,8,0.6855,0.0160,无可用信息(V<0.05),
3,payment_method,5,是否完成,订单,10996,4.9800,4,0.2894,0.0213,无可用信息(V<0.05),
4,shipping_method,3,订单状态,订单,10996,4.1500,8,0.8436,0.0137,无可用信息(V<0.05),
5,shipping_method,3,订单金额带,订单,10996,8.3100,8,0.4041,0.0194,无可用信息(V<0.05),
6,shipping_method,3,平台,订单,10996,0.6100,4,0.9616,0.0053,无可用信息(V<0.05),
7,shipping_method,3,是否完成,订单,10996,0.9700,2,0.6148,0.0094,无可用信息(V<0.05),
8,gender,2,订单状态,订单,10996,0.1200,4,0.9983,0.0033,无可用信息(V<0.05),
9,gender,2,订单金额带,订单,10996,1.7600,4,0.7804,0.0126,无可用信息(V<0.05),


无可用信息(V<0.05)的检验数： 18 / 24


## 模块 29｜阶段2 补充：复购间隔分布

In [36]:
# ============================================================
# 模块 29｜复购间隔分布（把「平均订单间隔」从均值升级为分布）
# 口径：已完成订单按用户、时间排序后的相邻订单间隔天数；仅 F>=2 的用户产生间隔对
# ============================================================
_cs = completed.sort_values(["global_user_id", "order_time"]).copy()
_cs["gap_days"] = _cs.groupby("global_user_id").order_time.diff().dt.days
_gap = _cs.dropna(subset=["gap_days"]).merge(
    S["rfm_detail"][["global_user_id", "segment"]], on="global_user_id", how="left")
_BINS = [0, 30, 60, 90, 180, 270, 360, np.inf]
_LBLS = ["0-30天", "30-60天", "60-90天", "90-180天", "180-270天", "270-360天", ">360天"]
_gap["gap_bin"] = pd.cut(_gap.gap_days, _BINS, labels=_LBLS)
_SEGS = ["高价值用户", "流失预警用户", "潜力用户", "低价值用户"]
_rows = []
for _seg in _SEGS:
    _s = _gap[_gap.segment == _seg]
    if len(_s) == 0:
        continue
    _cnt = _s.gap_bin.value_counts().reindex(_LBLS, fill_value=0)
    for _lb in _LBLS:
        _rows.append({"segment": _seg, "gap_bin": _lb, "pairs": int(_cnt[_lb]),
                      "pct_within_segment": round(100 * _cnt[_lb] / len(_s), 2)})
repurchase_interval_dist = pd.DataFrame(_rows)
repurchase_interval_dist.to_csv(MET / "repurchase_interval_dist.csv", index=False, encoding="utf-8-sig")
_sum = (_gap.groupby("segment").gap_days
        .agg(pairs="count", min="min", p25=lambda s: s.quantile(.25), median="median",
             p75=lambda s: s.quantile(.75), p90=lambda s: s.quantile(.90), max="max")
        .round(1).reset_index())
_sum["users_with_repurchase"] = _sum.segment.map(_gap.groupby("segment").global_user_id.nunique())
# 用户等权口径均值（与报告 3.4 节「平均订单间隔」同一算法：先按用户求均值，再按分层求均值），
# 与上表的订单对等权分位点口径不同，两者并列以说明口径差异
_user_mean = _gap.groupby(["segment", "global_user_id"]).gap_days.mean().groupby("segment").mean()
_sum["user_equal_weight_mean"] = _sum.segment.map(_user_mean).round(1)
_sum = _sum[["segment", "users_with_repurchase", "pairs", "min", "p25", "median", "p75", "p90", "max",
             "user_equal_weight_mean"]]
_sum.to_csv(MET / "repurchase_interval_summary.csv", index=False, encoding="utf-8-sig")
display(_sum)

# fig16：复购间隔分布 + 分位点
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
_cmap = {"高价值用户": "#2E86AB", "流失预警用户": "#F4A259", "潜力用户": "#5B8E7D", "低价值用户": "#BC4749"}
_xs = np.arange(len(_LBLS)); _wd = 0.8 / len(_SEGS)
for _i, _seg in enumerate(_SEGS):
    _d = repurchase_interval_dist[repurchase_interval_dist.segment == _seg].set_index("gap_bin").reindex(_LBLS)
    if len(_d) == 0:
        continue
    ax[0].bar(_xs + (_i - (len(_SEGS) - 1) / 2) * _wd, _d.pct_within_segment.fillna(0),
              width=_wd, label=_seg, color=_cmap[_seg])
ax[0].set_xticks(_xs); ax[0].set_xticklabels(_LBLS, rotation=20)
ax[0].set_ylabel("占该分层复购间隔对的比例（%）"); ax[0].set_title("RFM 分层 复购间隔分布")
ax[0].legend(fontsize=8)
_QIS = ["p25", "median", "p75", "p90"]; _x2 = np.arange(len(_SEGS)); _w2 = 0.2
for _j, _q in enumerate(_QIS):
    _v = [float(_sum.loc[_sum.segment == _s, _q].iloc[0]) if (_sum.segment == _s).any() else 0.0 for _s in _SEGS]
    _b = ax[1].bar(_x2 + (_j - 1.5) * _w2, _v, width=_w2, label=_q.upper())
    for _bb, _vv in zip(_b, _v):
        ax[1].text(_bb.get_x() + _bb.get_width() / 2, _vv + 3, f"{_vv:.0f}", ha="center", fontsize=7)
ax[1].set_xticks(_x2); ax[1].set_xticklabels(_SEGS, fontsize=9)
ax[1].set_ylabel("间隔天数（天）"); ax[1].set_title("复购间隔分位点（天）"); ax[1].legend(fontsize=8, ncol=4)
fig.suptitle("复购间隔分布（已完成订单口径，仅含 F>=2 用户的相邻订单对）", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG / "fig16_复购间隔分布.png", dpi=150, bbox_inches="tight"); plt.show()


,segment,users_with_repurchase,pairs,min,p25,median,p75,p90,max,user_equal_weight_mean
0,流失预警用户,93,114,1.0000,50.2000,112.0000,188.0000,273.8000,419.0000,135.7000
1,潜力用户,33,36,0.0000,97.2000,231.5000,390.8000,531.0000,661.0000,268.5000
2,高价值用户,231,423,0.0000,50.5000,119.0000,308.5000,475.6000,680.0000,254.0000


## 模块 30｜阶段4 补充：帕累托累计贡献曲线（CUME_DIST）

In [37]:
# ============================================================
# 模块 30｜帕累托累计贡献曲线（CUME_DIST 口径，用户 + SKU）
# 数据源：SQL 分段 pareto_cumulative（CUME_DIST() 计算名次分位）
# 提示：头部 20% 的切分定义仍由 pareto_user / pareto_sku 的 NTILE(5) 给出（人数可能相差 1）
# ============================================================
_pcd = S["pareto_cumulative"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for _sc, _lb, _c in [("用户", "用户（1643 名有效消费用户）", "#2E86AB"),
                     ("SKU", "SKU（2043 个动销 SKU）", "#E63946")]:
    _d = _pcd[_pcd.scope == _sc].sort_values("cut_pct")
    ax[0].plot(_d.cut_pct, _d.gmv_pct_cum, marker="o", label=_lb, color=_c, lw=2)
    for _xx, _yy in zip(_d.cut_pct, _d.gmv_pct_cum):
        if int(_xx) in (20, 50, 100):
            ax[0].annotate(f"{_yy:.2f}%", (_xx, _yy), textcoords="offset points",
                           xytext=(0, 6), ha="center", fontsize=8, color=_c)
ax[0].axvline(20, color="gray", ls="--", lw=1); ax[0].axhline(80, color="gray", ls=":", lw=1)
ax[0].text(21, 81, "经典 80/20 参照", fontsize=8, color="gray")
ax[0].set_xlabel("头部累计人数占比（%，按价值降序）"); ax[0].set_ylabel("累计贡献 GMV 占比（%）")
ax[0].set_title("帕累托累计贡献曲线（CUME_DIST 口径）"); ax[0].legend(fontsize=8, loc="lower right")
_d2 = _pcd[_pcd.scope == "用户"].sort_values("cut_pct")
ax[1].bar(_d2.cut_pct - 1.7, _d2.entity_pct, width=3.3, label="累计人数占比", color="#8D99AE")
ax[1].bar(_d2.cut_pct + 1.7, _d2.gmv_pct_cum, width=3.3, label="累计 GMV 占比", color="#2E86AB")
ax[1].set_xticks(_d2.cut_pct); ax[1].set_xlabel("头部累计分位节点（%）"); ax[1].set_ylabel("占比（%）")
ax[1].set_title("用户口径：累计人数占比 vs 累计 GMV 占比"); ax[1].legend(fontsize=8)
fig.suptitle("帕累托二阶视角：累计贡献曲线（头部 20% 切分仍由 NTILE(5) 定义，见 2.3 节）",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG / "fig17_帕累托累计贡献曲线.png", dpi=150, bbox_inches="tight"); plt.show()


## 模块 31｜流水线产物清单与交付检查

本格目标：校验六阶段全部产物齐备。分析结论见 `report/数据分析报告.md`，指标定义见 `report/指标字典.md`，Excel 第三腿复算步骤见根目录 `excel_cross_check.md`。

In [38]:
required = [
    "README.md", "excel_cross_check.md", "requirements.txt", ".gitignore", "LICENSE",
    "cross_check.xlsx",
    "code/data_process.sql", "code/pipeline.py", "code/analysis_pipeline.ipynb",
    "report/数据分析报告.md", "report/指标字典.md", "report/项目自检与口径备忘.md", "report/SQL亮点解读.md",
    "report/PowerBI搭建指南.md",
    "figures/fig01_分平台转化漏斗对比.png", "figures/fig02_Cohort同期群留存热力图.png",
    "figures/fig03_RFM用户分层占比饼图.png", "figures/fig04_各品类销售额柱状图.png",
    "figures/fig05_爆品与滞销SKU对比.png", "figures/fig06_月度GMV趋势折线图.png",
    "figures/fig07_平台会员等级转化率.png", "figures/fig08_EDA年龄与价格带转化.png",
    "bi_screenshots/bi_01_经营总览.png",
    "bi_screenshots/bi_02_转化漏斗.png",
    "bi_screenshots/bi_03_用户商品运营.png",
    "bi_screenshots/多平台电商经营看板.pbix",
    "bi_screenshots/说明.md",
    "figures/fig12_订单状态分布与平台构成.png",
    "figures/fig13_促销完成率与成交标价比率.png",
    "figures/fig14_品类共购lift与UPT.png",
    "figures/fig15_下单时段与星期结构.png",
    "figures/fig16_复购间隔分布.png",
    "figures/fig17_帕累托累计贡献曲线.png",
]
missing = [p for p in required if not (ROOT / p).exists()]
print("交付文件缺失：", missing if missing else "无，全部齐备")
print("SQL/增强中间结果：", len(list(MET.glob('*.csv'))), "个 CSV（output/metrics/）")
print("PowerBI 聚合集：", len(list(BIDIR.glob('*.csv'))), "个 CSV（output/bi/）")
print("图表：", len(list(FIG.glob('*.png'))), "张分析图 PNG（figures/）+ ",
      len(list((ROOT / 'bi_screenshots').glob('*.png'))), "张 PowerBI 看板截图（bi_screenshots/，手工产物）")
print("交叉校验：SQL 分段（code/data_process.sql）与 pandas 独立重算的 9 项核心指标全部一致"
      "（output/metrics/cross_check_sql_python.csv）；第三腿 Excel 复算结果缓存在根目录"
      " cross_check.xlsx，公式与操作说明见 excel_cross_check.md")


交付文件缺失： 无，全部齐备
SQL/增强中间结果： 54 个 CSV（output/metrics/）
PowerBI 聚合集： 15 个 CSV（output/bi/）
图表： 14 张分析图 PNG（figures/）+  3 张 PowerBI 看板截图（bi_screenshots/，手工产物）
交叉校验：SQL 分段（code/data_process.sql）与 pandas 独立重算的 9 项核心指标全部一致（output/metrics/cross_check_sql_python.csv）；第三腿 Excel 复算结果缓存在根目录 cross_check.xlsx，公式与操作说明见 excel_cross_check.md
